In [ ]:
import SimpleITK as sitk # https://simpleelastix.readthedocs.io/

import numpy as np
import scipy as sp
import pandas as pd
from scipy import ndimage
import scipy.interpolate
from scipy.spatial.transform import Rotation as R

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

import skimage
import skimage.io
import skimage.transform
import skimage.exposure
import skimage.filters
import skimage.morphology

import tifffile
import nrrd

import re, os, shutil, ast, sys, time, random, gc, math, pickle, time, copy

import anndata

%matplotlib inline

# Functions to display images
Functions originate from the [Allen Institute Whole Mouse Brain Atlas Analysis](https://github.com/ZhuangLab/whole_mouse_brain_MERFISH_atlas_scripts_2023/tree/main)

In [ ]:
# Functions to display images

# overlay to images to green and magenta
def imageoverlay(imG, imM):
    assert imG.shape == imM.shape
    output = np.zeros(imG.shape + (3,))
    g = skimage.util.img_as_float(imG)
    m = skimage.util.img_as_float(imM)
    output[:,:,0] = m 
    output[:,:,2] = m
    output[:,:,1] = g
    return output

# makes result image have values from 0-1 so it is acceptable to skimage
def scale_result(im):
    return (im - np.min(im))/np.ptp(im)

# adjust the contrast and brightness using percentage of cumulative distribution
def imagescPercent(im, percentlow, percenthigh):
    c , bins = skimage.exposure.cumulative_distribution(im)
    intlow = bins[np.argmin(np.abs(c - percentlow))].astype(im.dtype)
    inthigh = bins[np.argmin(np.abs(c - percenthigh))].astype(im.dtype)
    return skimage.exposure.rescale_intensity(im, in_range = (intlow, inthigh))

def border_transparency(im, RGBval = [1,1,1]):
    im = im/np.amax(im)
    alphas = im > np.amax(im)/5 # the input should have been a single values image anyways...
    rgb = im[:,:,None] * np.array(RGBval)
    return np.concatenate((rgb, alphas[:,:,None]), axis = 2)

# adjust the contrast and brightness using percentage of cumulative distribution
def adjustBC(im, percentlow, percenthigh):
    c , bins = skimage.exposure.cumulative_distribution(im)
    intlow = bins[np.argmin(np.abs(c - percentlow))].astype(im.dtype)
    inthigh = bins[np.argmin(np.abs(c - percenthigh))].astype(im.dtype)
    return skimage.exposure.rescale_intensity(im, in_range = (intlow, inthigh))

# rescale an image size and return uint16
# the preserve range thing here is weird, can it be set to true without the conversion?
def image_rescale(im, factor):
    return skimage.util.img_as_uint(skimage.transform.rescale(im, factor, preserve_range = False))

# fit a list of different sized images into one array
def image_list_to_array(ims):
    size_max = np.amax(np.array([im.shape for im in ims]), axis = 0)
    output = np.zeros([len(ims),size_max[0],size_max[1]], dtype = ims[0].dtype)
    for i,im in enumerate(ims):
        cols, rows = im.shape
        output[i,0:cols,0:rows] = im
    return output

# make some sort of funtion to pre process the data for registration

# gaussian blur
def blur(im, rad = 15):
    return skimage.filters.gaussian(im, rad)

def preprocess(im, thresh_factor = 1, rad = 15):
    edge = skimage.filters.sobel(im)
    thresh = skimage.filters.threshold_otsu(edge)
    return skimage.filters.gaussian(edge > thresh, rad)

def preprocess(im, thresh_factor = 1, rad = 5):
    edge = skimage.filters.sobel(im)
    thresh = skimage.filters.threshold_otsu(edge)
    sel = skimage.morphology.disk(rad)
    output =  blur(skimage.morphology.binary_closing(edge > (thresh * thresh_factor), sel))
    return np.clip(output,0,1)

# binarize an image after blurring and choosing a threshold
def binarize(im, blur_size = 1, thresh_factor = 0.1):
    blurred = blur(im, rad = blur_size) 
    thresh = thresh_factor * skimage.filters.threshold_otsu(blurred)
    return blurred > thresh

def plot_histogram(im, bins = 256, irange = (0,65535)):
    histogram, bin_edges = np.histogram(im, bins=bins, range=irange)
    plt.plot(bin_edges[0:-1], histogram)
    plt.ylim(0,np.amax(histogram[1:]))
    plt.show()

def find_corner(im, area_thresh = 100):
    imB = binarize(im)
    labels = skimage.measure.label(imB)
    rprops = skimage.measure.regionprops(labels)
    rprops = [p for p in rprops if p.area > area_thresh]
    rmin, cmin = np.amin(np.array([p.bbox for p in rprops]), axis = 0)[[0,1]]
    rmax, cmax = np.amax(np.array([p.bbox for p in rprops]), axis = 0)[[2,3]]
    return rmin, cmin, rmax, cmax

# crop an image using the find_corner function
def crop_image(im):
    bbox = find_corner(im)
    return im[bbox[0]:bbox[2], bbox[1]:bbox[3]]

# this will make a test grid the same size as the image
# may be useful to judge how disruptive the transformation is
def make_grid_image(im, grid_size = 220, pix_size = 50):
    dxy = int(grid_size / pix_size)
    output = np.zeros(im.shape)
    for r in range(0, im.shape[0], dxy):
        output[r] = 1
    for c in range(0, im.shape[1], dxy):
        output[:,c] = 1
    return output

def make_grid_image_3D(im, grid_size = 220, pix_size = 50):
    dxy = int(grid_size / pix_size)
    output = np.zeros(im.shape)
    for s in range(0, im.shape[0], dxy):
        output[s] = 1
    for r in range(0, im.shape[1], dxy):
        output[:,r] = 1
    for c in range(0, im.shape[2], dxy):
        output[:,:,c] = 1
    return output

## funtions for transforming points sets
# we have to do some manual transformations to get to the point where we then use the elastix/transformix transforms
# specifically, there is an initial rotation, possible reflection, 

def mat_trans(dx, dy):
    return np.array([[1,0,dx],[0,1,dy],[0,0,1]])

def mat_scale(sx, sy):
    return np.array([[sx,0,0],[0,sy,0],[0,0,1]])

def mat_rot(deg): #2D rotation matrix
    theta = np.deg2rad(deg)
    return np.array([[np.cos(theta),-np.sin(theta),0],[np.sin(theta),np.cos(theta),0],[0,0,1]])

def mat_reflect(reflect = False, dx = 0):
    if reflect:
        return np.array([[-1,0,dx],[0,1,0],[0,0,1]])
    else:
        return np.array([[1,0,0],[0,1,0],[0,0,1]])

    
#adapted from skimage.transform.rotate to account for image size increase
def find_new_corner(rows, cols, deg):

    corners = np.array([
            [0, 0, 1],
            [0, rows - 1, 1],
            [cols - 1, rows - 1, 1],
            [cols - 1, 0, 1]
        ])
    
    # meant for odd numbers? regardless, this is the definition in skimage...
    center = [cols/2 - 0.5, rows/2 - 0.5]
    
    tform1 = mat_trans(-center[0], -center[1])
    tform2 = mat_rot(deg)
    tform3 = mat_trans(center[0], center[1])
    tform = tform3 @ tform2 @ tform1
    #print(corners) # these are the corners before transform
    corners = tform[None, :,:] @ corners[:,:,None]
    #print(corners) # these are the corners after transform
    minc = corners[:, 0].min()
    minr = corners[:, 1].min()
    maxc = corners[:, 0].max()
    maxr = corners[:, 1].max()
    
    out_rows = int(maxr - minr + 1)
    out_cols = int(maxc - minc + 1)
    
    # return the new corners and widths after rotation
    return [minr, minc, out_rows, out_cols]

def transform_positions(tform, array):
    vals = np.concatenate([array,np.ones((len(array),1))], axis = 1)
    out = tform[None,:,:] @ vals[:,:,None]
    return out[:,0:2,0]


# note change to index here
def write_pts_file(array, name = 'points.pts'):
    with open(name, 'w') as f:
        f.write('point\n')
        f.write(str(len(array)) + '\n')
        # write the values in x, y order
        for row in array:
            for r in row:
                f.write(str(r) + ' ')
            f.write('\n')
            
def read_outputpoints_file():
    # the order here is 
    # InputIndex InputPoint OutputIndexFixed OutputPoint Deformation OutputIndexMoving
    with open('outputpoints.txt', 'r') as f:
        lines = f.readlines()
    output = []
    for line in lines:
        columns = line.split('\t;')[1:]
        pairs = [list(map(float, col.split('=')[1].strip(' []\n').split( ))) for col in columns]
        output.append(pairs)
    return np.array(output)

# find the center of the raw image
# do we need to worry about whether the image dimensions are even or odd?
# maybe for large images it doesnt matter since one pixel off will be small
def find_center(im):
    imdim = im.shape
    
    if imdim[0]%2 == 0:
        imcenterr = imdim[0]/2
    else:
        imcenterr = imdim[0]/2 - 0.5

    if imdim[1]%2 == 0:
        imcenterc = imdim[1]/2
    else:
        imcenterc = imdim[1]/2 - 0.5
    
    return np.array([imcenterr,imcenterc])
    
    #return np.array(im.shape).astype(float)/2 - 0.5
    
# define function for parsing using the above regex - make sure it works
def parse_filename(filename, regex = None):
    if regex is None:
        regex = "(?P<sample>.*)_slice(?P<slice>\d)_y_(?P<y1>[-]*\d+)_(?P<y2>[-]*\d+)_x_(?P<x1>[-]*\d+)_(?P<x2>[-]*\d+)_(?P<remainder>.*)"
    pat = re.compile(regex)
    res = pat.match(filename)
    return [res['sample'],int(res['slice']),int(res['y1']),int(res['y2']),int(res['x1']),int(res['x2'])]

## functions for running elastix

def crop_and_pad_image(image1, image2 = None, pad_width = 20, area_thresh = 20):

    bbox = find_corner(image1, area_thresh = area_thresh)
    image1 = np.pad(image1[bbox[0]:bbox[2],bbox[1]:bbox[3]], pad_width)

    if not image2 is None:
        image2 = np.pad(image2[bbox[0]:bbox[2],bbox[1]:bbox[3]], pad_width)

    return bbox, image1, image2

def crop_and_pad_points(pointsXY, bbox, pad_width = 20):
    output_pts = pointsXY - np.array(bbox[1], bbox[0]) # XY order for sitk
    output_pts = output_pts + pad_width
    return output_pts
    
# def register_images(fixed_image, moving_image, params_rigid, params_spline = None):
#     elastixImageFilter = sitk.ElastixImageFilter()
#     elastixImageFilter.LogToFileOn()
    
#     elastixImageFilter.SetParameterMap(params_rigid)
#     if not params_spline is None:
#         elastixImageFilter.AddParameterMap(params_spline)
        
#     elastixImageFilter.SetFixedImage(sitk.GetImageFromArray(fixed_image))
#     elastixImageFilter.SetMovingImage(sitk.GetImageFromArray(moving_image))
#     elastixImageFilter.Execute()
    
#     result_image = sitk.GetArrayFromImage(elastixImageFilter.GetResultImage())
#     trans = elastixImageFilter.GetTransformParameterMap()
    
#     return trans, result_image

def register_images(fixed_image, moving_image, params_rigid, params_spline = None, fixed_mask = None):
    elastixImageFilter = sitk.ElastixImageFilter()
    
    elastixImageFilter.SetOutputDirectory
    
    elastixImageFilter.SetParameterMap(params_rigid)
    if not params_spline is None:
        elastixImageFilter.AddParameterMap(params_spline)
        
    elastixImageFilter.SetFixedImage(sitk.GetImageFromArray(fixed_image))
    elastixImageFilter.SetMovingImage(sitk.GetImageFromArray(moving_image))
    
    if 'CorrespondingPointsEuclideanDistanceMetric' in [item for sublist in params_rigid.asdict().values() for item in sublist]:
        print('load corresponding points')
        elastixImageFilter.SetFixedPointSetFileName("fix.pts")
        elastixImageFilter.SetMovingPointSetFileName("mov.pts")
    
    # this is not working
    if not fixed_mask is None:
        mask = sitk.GetImageFromArray(fixed_mask)
        mask = sitk.Cast(mask, sitk.sitkUInt8)
        elastixImageFilter.SetFixedMask(mask)
        print('fixed mask set')
    
    elastixImageFilter.Execute()
    
    result_image = sitk.GetArrayFromImage(elastixImageFilter.GetResultImage())
    trans = elastixImageFilter.GetTransformParameterMap()
    
    return trans, result_image

def transform_image(moving_image, params, interpolation = True, spacing = None):
    transformixImageFilter = sitk.TransformixImageFilter()
    
    if not interpolation:
        if isinstance(params, tuple):
            for param in params:
                param["ResampleInterpolator"] = L2P(["FinalNearestNeighborInterpolator"])
        else:
            params["ResampleInterpolator"] = L2P(["FinalNearestNeighborInterpolator"])
    
    transformixImageFilter.SetTransformParameterMap(params)

    mov = sitk.GetImageFromArray(moving_image)

    if not spacing is None:
        mov.SetSpacing(spacing)
        print('SITK spacing set to {}'.format(mov.GetSpacing()))
    
    transformixImageFilter.SetMovingImage(mov)
    transformixImageFilter.Execute()
    result = sitk.GetArrayFromImage(transformixImageFilter.GetResultImage())
    return result

#convert a param to a numpy list to do manipulations on it
def P2L(param):
    return np.asarray([float(p) for p in param])

#convert a numpy list to a tuple with strings for the params
def L2P(paramlist):
    strs = [str(p) for p in paramlist]
    return tuple(strs)

def params_from_df(df, num):     
    # make some affine params
    p = sitk.GetDefaultParameterMap("affine")
    p['NumberOfResolutions'] = L2P([4])
    p['MaximumNumberOfIterations'] = L2P([500])
    #p['NumberOfSpatialSamples'] = L2P([4096])
    p['NumberOfHistogramBins'] = L2P([32]) 
    if not np.isnan(df.iloc[num].histogram_bins):
        p['NumberOfHistogramBins'] = L2P([df.iloc[num].histogram_bins])
    # make some spline params
    p2 = sitk.GetDefaultParameterMap("bspline")
    #p2['NumberOfSpatialSamples'] = L2P([32000])
    p2['NumberOfResolutions'] = L2P([4])
    p2['GridSpacingSchedule'] = L2P([20,10,5,2])
    p2['NumberOfHistogramBins'] = L2P([32]) 
    if not np.isnan(df.iloc[num].histogram_bins):
        p2['NumberOfHistogramBins'] = L2P([df.iloc[num].histogram_bins])
    p2['MaximumNumberOfIterations'] = L2P([500])
    if not np.isnan(df.iloc[num].iterations):
        p2['MaximumNumberOfIterations'] = L2P([df.iloc[num].iterations])
    p2['FinalGridSpacingInPhysicalUnits'] = L2P([]) # this remove the param?
    p2['FinalGridSpacingInVoxels'] = L2P([16])
    if not np.isnan(df.iloc[num].spline_grid_size):
        p2['FinalGridSpacingInVoxels'] = L2P([df.iloc[num].spline_grid_size])
        
    # check if there is a csv file with corresponding points...crop_and_pad_image
    
    csv_file = os.path.splitext(df.iloc[num].Filename)[0] + '.csv'
    if os.path.exists(csv_file):
        print('corresponding points file found: {}'.format(csv_file))
        p['Registration'] = L2P(["MultiMetricMultiResolutionRegistration"])
        p['Metric'] = L2P(['AdvancedMattesMutualInformation','CorrespondingPointsEuclideanDistanceMetric'])
        p['Metric0Weight'] = L2P([1 - df.iloc[num].cor_pts_weight])
        p['Metric1Weight'] = L2P([df.iloc[num].cor_pts_weight])
        
        p2['Registration'] = L2P(["MultiMetricMultiResolutionRegistration"])
        p2['Metric'] = L2P(['AdvancedMattesMutualInformation','CorrespondingPointsEuclideanDistanceMetric'])
        p2['Metric0Weight'] = L2P([1 - df.iloc[num].cor_pts_weight])
        p2['Metric1Weight'] = L2P([df.iloc[num].cor_pts_weight])
    
    return p, p2

# importing images from a dataframe
# read the image with some info from a dataframe
def image_from_df(df, index, right_crop = True, scale = True):
    im = skimage.io.imread(df.iloc[index]['Filename'])
    if len(im.shape) == 3:
        im = im[:, :, 0]
    if not np.isnan(df.iloc[index]['rot_init']):
        im = skimage.transform.rotate(im, df.iloc[index]['rot_init'], resize = True)
    if df.iloc[index]['reflect']:
        im = np.flip(im, axis = 1)
    if right_crop:
        if not np.isnan(df.iloc[index]['right_crop']):
            im[:,int(df.iloc[index]['right_crop']):] = 0
    if scale:
        if not (np.isnan(df.iloc[index]['scale_x']) and np.isnan(df.iloc[index]['scale_y'])):
            new_size = im.shape * np.array([df.iloc[index]['scale_y'],df.iloc[index]['scale_x']])
            new_size = new_size.astype(int)
            im = skimage.transform.resize(im, new_size)
    return im

# ADDED FUNC
def safe_literal_eval(s):
    if pd.isna(s):  # Check if the value is nan
        return None  # Treat nan as None
    try:
        return ast.literal_eval(s)
    except ValueError as e:
        # Print a message for errors other than 'nan'
        if str(s) != 'nan':
            print(f"Failed to parse: {s} with error: {e}")
        return None  # Return None or some default value if parsing fails

def import_df(path):
    df = pd.read_excel(path)
    df = df.sort_values('z_pos', ignore_index = True)
    
    columns = ['allen_slice_num', 'angle_ZY', 'angle_ZX']    # columns to interpolate
    for col in columns:
        x = df.z_pos.values[np.logical_not(np.isnan(df[col].values))]
        y = df[col].values[np.logical_not(np.isnan(df[col].values))]
        f = scipy.interpolate.interp1d(x, y, kind='linear', fill_value = 'extrapolate')
        df[col] = f(df.z_pos)
    
    df['allen_slice_num'] = df['allen_slice_num'].values.astype(int)
    #
    df['space_modules'] = df['space_modules'].apply(lambda x: safe_literal_eval(str(x)))
    df['cell_types'] = df['cell_types'].apply(lambda x: safe_literal_eval(str(x)))
    df['annots_to_amplify'] = df['annots_to_amplify'].apply(lambda x: safe_literal_eval(str(x)))
    return df

# use the image info dataframe to get some cell metadata from the cell metadata dataframe

# for interactively selecting corresponding points
def onpick(event):
    xmouse = event.mouseevent.xdata
    ymouse = event.mouseevent.ydata
    
    points_cor_x.append(xmouse)
    points_cor_y.append(ymouse)
    
    if len(points_cor_x)%2!=0:
        axs[0].scatter(xmouse, ymouse , marker = 'x', color = 'y', s = 100, linewidth = 3)
        axs[0].annotate(str(math.ceil(len(points_cor_x)/2)), 
                        xy = [xmouse, ymouse], color = 'y', fontsize = 12)
    else:
        axs[1].scatter(xmouse, ymouse , marker = 'x', color = 'y', s = 100, linewidth = 3)
        axs[1].annotate(str(math.ceil(len(points_cor_x)/2)), 
                xy = [xmouse, ymouse], color = 'y', fontsize = 12)
    plt.draw()
    
# make certain regions of the dapi have more contrast using cell type information 

def modify_dapi(df, num, cmd, cell_types = None, space_modules = None, ccf_pixel_size = 25, factor = 10):
    dapi = image_from_df(df, num)
    cells = get_cell_metadata_for_slice_index(df, num, cmd, ccf_pixel_size = ccf_pixel_size)

    mean_val = np.mean(dapi[dapi > 0])

    ymax,xmax = dapi.shape[:2]
    mask = np.zeros(dapi.shape)

    cells = get_cell_metadata_for_slice_index(df, num, cmd, ccf_pixel_size = ccf_pixel_size)
    x = np.round(cells.fixed_x).astype(int)
    y = np.round(cells.fixed_y).astype(int)

    cond = ((x > 0) & (x < (xmax -1)) &
            (y > 0) & (y < (ymax -1)))
    cells = cells[cond]

    # enhance cell types
    if not cell_types is None:
        mask2 = np.zeros(dapi.shape)
        df_temp = cells[cells.subclass_label_transfer.isin(cell_types)]
        x = np.round(df_temp.fixed_x).astype(int)
        y = np.round(df_temp.fixed_y).astype(int)
        mask2[y,x] = mean_val * factor
        mask2 = skimage.filters.gaussian(mask2, sigma = 1)
        dapi += mask2
        
    # enhance space modules
    if not space_modules is None:
        mask2 = np.zeros(dapi.shape)
        df_temp = cells[cells.spatial_modules_level_1_name.isin(space_modules)]
        x = np.round(df_temp.fixed_x).astype(int)
        y = np.round(df_temp.fixed_y).astype(int)
        mask2[y,x] = mean_val * factor
        mask2 = skimage.filters.gaussian(mask2, sigma = 1)
        dapi += mask2

    # remove any modifications after 'right crop'
    if not np.isnan(df.iloc[num]['right_crop']):
        rc = df.iloc[num]['right_crop']
        sx = df.iloc[num]['scale_x']
        rc_new = int(np.round(rc * sx))
        dapi[:,rc_new:] = 0
    
    return dapi

Function for pulling cell metadata from slice

In [ ]:
# use the image info dataframe to get some cell metadata from the cell metadata dataframe

def get_cell_metadata_for_slice_index(df, index, cmd, ccf_pixel_size = 25, bbox = None, pad_width = 20):
    file = df.iloc[index].Filename
    rot = df.iloc[index].rot_init
    reflect = df.iloc[index].reflect
    scale_x, scale_y = [df.iloc[index].scale_x, df.iloc[index].scale_y]
    
    # here is the cell metadata subset
    cmd_temp = cmd[cmd.sample_id == df.iloc[index].cell_metadata]
    
    y1 = min(cmd_temp["center_y"])
    y2 = max(cmd_temp["center_y"])
    x1 = min(cmd_temp["center_x"])
    x2 = max(cmd_temp["center_x"])

    # need to know some stuff about the raw image
    imraw = skimage.io.imread(file)
    imdim = np.array(imraw.shape)
    imcenter = find_center(imraw)
    umperpixY = (y2-y1)/imdim[0]
    umperpixX = (x2-x1)/imdim[1]
    
    
    
    tform1 = mat_trans(-x1,-y1) # move the corner of the image to the origin
    tform2 = mat_scale(1/umperpixX, 1/umperpixY) # scale the image into pixels
    tform3 = mat_trans(imcenter[1],imcenter[0]) @ mat_rot(-rot) @ mat_trans(-imcenter[1],-imcenter[0]) # do initial rotation (translated to the center, rotate, translate back)
    shiftr, shiftc, out_rows, out_cols = find_new_corner(imdim[0], imdim[1], -rot) # find the new size of the image since it was rotated
    tform4 = mat_trans(-shiftc,-shiftr) # shift to the new corner
    tform5 = mat_reflect(reflect = reflect, dx = out_cols) # reflect if there is a reflection
    tform6 = mat_scale(scale_x,scale_y) # do final scaling in x and y which was added for better registration
    if bbox is None:
        tform =  tform6 @ tform5 @ tform4 @ tform3 @ tform2 @ tform1 # final transform of everything!
    else:
        tform7 = mat_trans(-bbox[1],-bbox[0]) # crop in on the cells
        tform8 = mat_trans(pad_width,pad_width) # add padding
        tform =  tform8 @ tform7 @ tform6 @ tform5 @ tform4 @ tform3 @ tform2 @ tform1 # final transform of everything!
    
    cells =  cmd_temp[(y1 < cmd_temp['center_y']) &
                      (cmd_temp['center_y'] < y2) & 
                      (x1 < cmd_temp['center_x']) & 
                      (cmd_temp['center_x'] < x2)].copy() # make a copy so it does not give the slice warning...
    
    cells_pos = np.array([cells['center_x'], cells['center_y']]).T
    cells_pos_image_space = transform_positions(tform, cells_pos) # transform the cell positions from the initial transforms
    cells['fixed_x'] = cells_pos_image_space[:,0]
    cells['fixed_y'] = cells_pos_image_space[:,1]
    return cells

In [ ]:
# check if elastix is working
elastixImageFilter = sitk.ElastixImageFilter()

In [ ]:
# Isolated optimization copy: all mutable outputs are redirected away from the original workflow.
base_path = "/resnick/groups/mthomson/jboktor/WILDRxSPF_brains"
ref_path = os.path.join(base_path, 'data/input/allen_registration_ref')
analysis_reg_path = os.path.join(base_path, 'data/interim/registration/Allen_CCF_optimization')
iteration_root = os.path.join(base_path, 'figures/Allen_CCF_alignment_optimized')
os.makedirs(analysis_reg_path, exist_ok=True)
os.makedirs(iteration_root, exist_ok=True)
os.chdir(analysis_reg_path)  # isolate transformix/elastix temporary files from the original notebook directory
base_path

# subclass color csv file
# subclass_color_csv_file = os.path.join(ref_path, 'subclass_colors_new.csv')

# allen lut csv file
allen_lut_file = os.path.join(ref_path, 'allen_lut.csv')
# dictionary mapping allen name to annotation
name_to_annotation_file = os.path.join(ref_path, 'allen_name_to_annots.pkl')

# allen nissl and annotation file
# http://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/ 
# & https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/ara_nissl/
ara_nissl_file = os.path.join(ref_path, 'ara_nissl_25.nrrd')
ara_annot_file = os.path.join(ref_path, 'annotation_25.nrrd')
ara_annot_10_file = os.path.join(ref_path, 'annotation_10.nrrd')
# allen border file
# generated in "allen annotation to border.ipynb" 
allen_border_file = os.path.join(ref_path, 'annotations_25_border_only.tif')


In [ ]:
# this excel sheet stores info about the dapi slice images and registration parameters
slice_info_filename = os.path.join(analysis_reg_path, 'slice_positions_25um_final.xlsx')

# this excel sheet store correspondence between cell types and annotated regions
structure_df_file = os.path.join(ref_path, 'cell_types_to_annotations_adjusted.xlsx')

# cell metadata file
# key columns are center_x, center_y, slice_id, subclass_label_transfer, spatial_modules_level_1
cmd_file = os.path.join(ref_path, 'cell-metadata_2026-02-27.csv')


# Importing images from dataframe

In [ ]:
# import the excel sheet that has the slice positions, rotations and reflections
# sort depending on the Z value
df = pd.read_excel(slice_info_filename)
df = df.sort_values('z_pos', ignore_index = True)
df

Visualizing the DAPI images

In [ ]:
# for i in range(len(df)):
#     plt.figure()
#     img = image_from_df(df, i, scale = False)
#     print(img.shape)
#     plt.imshow(adjustBC(img, 0, 0.7), cmap = 'gray')
#     plt.show()

# Generating colormaps

In [ ]:
# rainbow colormap
# take the prism colormap and make the zero value black
# turn it into a listed colormap
cmap = cm.prism(np.arange(256))
cmap[0] = [0,0,0,1]
cmap = ListedColormap(cmap)
# set maximum colors to display
max_colors = 1140

cmap

In [ ]:
# make an LUT with unique values

def RGB_lut_255(unique_values):
    cmap = (255 * cm.gist_rainbow(np.arange(256))).astype(np.uint8)
    cmap[0] = [0,0,0,1]
    return cmap[np.linspace(0,255,unique_values).astype(int)][:,0:3]

In [ ]:
# Allen colormap
df_lut = pd.read_csv(allen_lut_file, names = ['Annotation','R','G','B'])

# make a blank look up table
# this is actually a numpy list
lut = np.zeros([np.amax(df_lut.Annotation) + 1,3], dtype = np.uint8) # add 1 to max value...
# lut needs to be uint8 for RGB
lut.shape

# set the values from the lut
for i,row in df_lut.iterrows():
    lut[row.Annotation] = [row.R,row.G,row.B]

In [ ]:
# # new subclass colors
# temp_df = pd.read_csv(subclass_color_csv_file)
# subclass_color = {}
# for i, row in temp_df.iterrows():
#     subclass_color[row.subclass] = row.color
    
# subclass_color['LQ'] = '#ffffff'

# # use old DG Glut color
# subsubclass_color['DG Glut'] = '#2bb179'


# Importing Allen Reference Data

In [ ]:
#have all of the allen name to annotations at hand
with open(name_to_annotation_file, 'rb') as handle:
    allen_name_to_annots = pickle.load(handle)

len(allen_name_to_annots)

Importing nissl

In [ ]:
ccf_pixel_size = 25
midline = 228
extra = 8

nissl, header = nrrd.read(ara_nissl_file, index_order='F')
nissl = nissl[:,:,:midline + extra]
print(nissl.shape)

# import annotations
annot, header = nrrd.read(ara_annot_file, index_order='F')
annot = annot[:,:,0:midline + extra]
print(annot.shape)

In [ ]:
# having some trouble interpolating the annotated image with high values
# hopefully this can shift the values to unused lower values and then shift back the original values
annot_unique = np.unique(annot)

annot_convert = np.zeros(annot_unique[-1] + 1, dtype=np.uint32)
for i,val in enumerate(annot_unique):
    annot_convert[val] = i

annot_revert = np.zeros(len(annot_unique), dtype=np.uint32)
for i,val in enumerate(annot_unique):
    annot_revert[i] = val

Sample visualization of Allen Reference Atlas (ARA) data

In [ ]:
# for slice_num in range(100, 451, 50):
#     fig, axs = plt.subplots(1,3, figsize = (10,5))
#     axs[0].imshow(lut[annot[slice_num]])
#     axs[1].imshow(annot_convert[annot[slice_num]])
#     axs[2].imshow(lut[annot_revert[annot_convert[annot[slice_num]]]])


Reading in Allen Boarders Data

In [ ]:
# import borders
borders = skimage.io.imread(allen_border_file)
borders = borders[:,:,:midline + extra]
print(borders.shape)
print(borders.dtype)

In [ ]:
slice_num = 300
fig, axs = plt.subplots(1,4, figsize = (10,5))
axs[0].set_title(slice_num)
axs[0].imshow(nissl[slice_num], cmap = 'gray')
# axs[1].imshow(lut[annot[slice_num]])
axs[1].imshow(borders[slice_num], alpha = 0.6)
axs[2].imshow(lut[annot[slice_num]])
axs[3].imshow(nissl[slice_num], cmap = 'gray')
axs[3].imshow(lut[annot[slice_num]], alpha = 0.4)


In [ ]:
for slice_num in range(250, 285, 5):
    fig, axs = plt.subplots(1,4, figsize = (10,5))
    axs[0].set_title(slice_num)
    axs[0].imshow(nissl[slice_num], cmap = 'gray')
    # axs[1].imshow(lut[annot[slice_num]])
    axs[1].imshow(borders[slice_num], alpha = 0.6)
    axs[2].imshow(lut[annot[slice_num]])
    axs[3].imshow(nissl[slice_num], cmap = 'gray')
    axs[3].imshow(lut[annot[slice_num]], alpha = 0.4)


# Reading in Spatial Genomics Slice Data

Slice metadata

In [ ]:
# # slice_info_filename_new = 'WB3_Coronal_1_mosaic_positions_25um_final_EDITED.xlsx'
# df.to_excel(slice_info_filename, index=False)
df = pd.read_excel(slice_info_filename)
df

Cell metadata

In [ ]:
cmd = pd.read_csv(cmd_file, index_col = 0)
cmd.head()

In [ ]:
allen_name_to_annots.keys()

In [ ]:
# make certain regions of nissl have more contrast

def modify_nissl(nissl, annot, factor = 10, annot_dict = None, midline = 228):
    output = np.copy(nissl)
    mean_val = np.mean(output[output > 0])
    annot = np.rint(annot).astype(np.int32)

    # vlmc layer
    mask = annot > 0
    output = output * mask # clear outside the annotation
    
    surface = np.logical_xor(mask, skimage.morphology.binary_dilation(mask, skimage.morphology.disk(2)))
    output[surface] = mean_val * factor # add vlmc
    
    # vlmc layer at SM_CTX/olf
    mask = np.isin(annot, allen_name_to_annots['Isocortex'])
    surface = np.logical_xor(mask, skimage.morphology.binary_dilation(mask, skimage.morphology.disk(2)))    
    output[surface] = mean_val * factor # add vlmc
    
    # vlcm at midline
    mask_ctx = np.isin(annot, allen_name_to_annots['Cerebral cortex'] + allen_name_to_annots['olfactory nerve layer of main olfactory bulb'])
    mask_midline = np.zeros(annot.shape, dtype = bool)
    mask_midline[:,midline-1:midline + 2] = True
    mask_midline = np.logical_and(mask_ctx, mask_midline)
    output[mask_midline] = mean_val * factor

    
    # Ependymal NN
    #annots_ENN = [73, 81, 89, 98, 108, 116, 124, 129, 140, 145, 153, 164]
    annots_ENN = [81,129]
    ventrical_mask = np.isin(annot, annots_ENN)
    edge_mask = np.logical_xor(ventrical_mask, skimage.morphology.binary_erosion(ventrical_mask, skimage.morphology.disk(2)))
    
    output[ventrical_mask] = 0 # remove ventricals
    output[edge_mask] = mean_val * factor # add surface
    
    
    if not annot_dict is None:
        for val in annot_dict:
            mask = (annot == val)
            output += mask * mean_val * factor
            
    return output

In [ ]:
annots_to_amplify = [507, 698, 665, 538, 900] + [632]
annots_to_amplify = annots_to_amplify + allen_name_to_annots['Isocortex']

allen_slice_num = 280

output = modify_nissl(
                nissl[allen_slice_num],
                annot[allen_slice_num],
                annot_dict = annots_to_amplify,
                factor = 2)

fig, axs = plt.subplots(1,1)
axs.imshow(output, cmap = 'gray')

# modify dapi

In [ ]:
cmd.head()

In [ ]:
# print(cmd["singleR_labels"].unique())
# ['29 CB Glut' '06 CTX-CGE GABA' '34 Immune' '07 CTX-MGE GABA'
#  '33 Vascular' '30 Astro-Epen' '09 CNU-LGE GABA' '01 IT-ET Glut'
#  '12 HY GABA' '11 CNU-HYa GABA' '20 MB GABA' '05 OB-IMN GABA' '19 MB Glut'
#  '26 P GABA' '14 HY Glut' '13 CNU-HYa Glut' '24 MY Glut' '10 LSX GABA'
#  '02 NP-CT-L6b Glut' '27 MY GABA' '08 CNU-MGE GABA' '25 Pineal Glut'
#  '16 HY MM Glut' '18 TH Glut' '04 DG-IMN Glut' '31 OPC-Oligo' '23 P Glut'
#  '17 MH-LH Glut' '21 MB Dopa' '03 OB-CR Glut' '28 CB GABA' '32 OEC']

print(np.sort(cmd["class_name"].unique()))

In [ ]:
# cell_types_to_amplify = ['01 IT-ET Glut', 
#                          '02 NP-CT-L6b Glut', 
#                          '04 DG-IMN Glut',
#                          '07 CTX-MGE GABA',
#                         '33 Vascular',
#                          '11 CNU-HYa GABA']

cell_types_to_amplify = [
    "001 CLA-EPd-CTX Car3 Glut",
    # "002 IT EP-CLA Glut",
    # "003 L5/6 IT TPE-ENT Glut",
    "004 L6 IT CTX Glut",
    "005 L5 IT CTX Glut",
    "006 L4/5 IT CTX Glut",
    "007 L2/3 IT CTX Glut",
    "016 CA1-ProS Glut",
    "017 CA3 Glut",
    "020 L2/3 IT RSP Glut",
    "021 L4 RSP-ACA Glut",
    "023 SUB-ProS Glut",
    "025 CA2-FC-IG Glut",
    "026 NLOT Rho Glut",
    # "027 L6b EPd Glut",
    "029 L6b CTX Glut",
    "030 L6 CT CTX Glut",
    "032 L5 NP CTX Glut",
    "033 NP SUB Glut",
    "037 DG Glut",
    "329 ABC NN",
    "330 VLMC NN",
    "332 SMC NN",
]

#  [1] "330 VLMC NN"               "032 L5 NP CTX Glut"       
#  [3] "001 CLA-EPd-CTX Car3 Glut" "005 L5 IT CTX Glut"       
#  [5] "029 L6b CTX Glut"          "017 CA3 Glut"             
#  [7] "025 CA2-FC-IG Glut"        "026 NLOT Rho Glut"        
#  [9] "329 ABC NN"                "004 L6 IT CTX Glut"       
# [11] "006 L4/5 IT CTX Glut"      "023 SUB-ProS Glut"        
# [13] "332 SMC NN"                "016 CA1-ProS Glut"        
# [15] "020 L2/3 IT RSP Glut"      "007 L2/3 IT CTX Glut"     
# [17] "002 IT EP-CLA Glut"        "030 L6 CT CTX Glut"       
# [19] "003 L5/6 IT TPE-ENT Glut"  "021 L4 RSP-ACA Glut"      
# [21] "037 DG Glut"               "027 L6b EPd Glut"         
# [23] "033 NP SUB Glut"          


In [ ]:
num  = 7
modify_dapi(df, num, cmd, cell_types = cell_types_to_amplify, factor = 10)

In [ ]:
for num in range(len(df)):
    df = pd.read_excel(slice_info_filename)
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    # DAPI with contrast
    fixed_image = imagescPercent(image_from_df(df, num, scale=False), 0, 0.6)
    axs[0].imshow(adjustBC(fixed_image, 0, 0.8), cmap='gray', aspect='equal')
    axs[0].set_title('DAPI w/ contrast')

    # DAPI with amplified cell type signal
    dapi_mod = modify_dapi(df, num, cmd, cell_types = cell_types_to_amplify, factor = 10)
    axs[1].imshow(dapi_mod, cmap = 'gray', vmin = 0)
    axs[1].set_title('New DAPI cells highlighted')

    plt.tight_layout()
    plt.subplots_adjust(wspace=0)
    plt.show()

# 2D Registration

In [ ]:
# test the registration of a slice
# parameters for registration are stored in the slice_info_filename
# an interactive popup window will appear
# alternately click on left image then right image to record corresponding points
# run the following cell to save a csv file
# the code will automatically find a csv file if it is present

slice_info_filename

In [ ]:
slice_to_register = 11
print('slice to register {}'.format(slice_to_register))

num = slice_to_register - 1

df = pd.read_excel(slice_info_filename)
ccf_pixel_size = 25

filename = df.iloc[num].Filename
print(filename)

allen_slice_num = df.iloc[num].allen_slice_num
print(allen_slice_num)

rescale_percent = 0.99

moving_annot = annot[allen_slice_num]
moving_borders = borders[allen_slice_num]
nissl_im = nissl[allen_slice_num]

print('nissl factor {}'.format(df.iloc[num].nissl_enhance_factor))
print('dapi factor {}'.format(df.iloc[num].dapi_enhance_factor))

fixed = modify_dapi(df, num, cmd, cell_types = cell_types_to_amplify, factor = 2)
# fixed = modify_dapi(df, num, cmd,
#     cell_types = df.iloc[num].cell_types,
#     space_modules = df.iloc[num].space_modules,
#     factor = df.iloc[num].dapi_enhance_factor)


moving  = modify_nissl(nissl_im, moving_annot)

pad_width = 20
area_thresh = df.iloc[num].area_thresh

# crop images (may not be necessary since we did it already) and add a pad
fix_bbox, fixed, _ = crop_and_pad_image(fixed, pad_width = pad_width, area_thresh = area_thresh)
_, _, moving_borders = crop_and_pad_image(moving, moving_borders, pad_width = pad_width, area_thresh = area_thresh) # be careful here don't overwrite the original moving...
mov_bbox, moving, moving_annot = crop_and_pad_image(moving, moving_annot, pad_width = pad_width, area_thresh = area_thresh) 

cells = get_cell_metadata_for_slice_index(df, num, cmd, ccf_pixel_size = ccf_pixel_size, bbox = fix_bbox, pad_width = pad_width)


fig, axs = plt.subplots(1,2)
axs[0].imshow(fixed, cmap = 'gray', picker=False)
axs[0].set_facecolor((0,0,0))
axs[0].scatter(cells['fixed_x'], cells['fixed_y'], s = 0.3, c = cells.subclass_color)

#axs[1].imshow(lut[moving_annot],  picker=True)
axs[1].imshow(moving,  picker=True)
axs[1].imshow(moving_borders, alpha = 0.5, cmap = 'gray')

fig.suptitle(filename)

points_cor_x = []
points_cor_y = []

fig.canvas.mpl_connect('pick_event', onpick)

In [ ]:
# if this cell is run, the points will be saved (or overwritten)
filename_csv = os.path.splitext(filename)[0] + '.csv'

df_points = pd.DataFrame()
df_points['fix_x'] = np.array(points_cor_x)[::2]
df_points['fix_y'] = np.array(points_cor_y)[::2]
df_points['mov_x'] = np.array(points_cor_x)[1::2]
df_points['mov_y'] = np.array(points_cor_y)[1::2]

# df_points.to_csv(filename_csv, index=False)

print(filename_csv)
df_points

In [ ]:
# annots_to_amplify = []

In [ ]:
# make some params
p, p2 = params_from_df(df, num)
# crop and pad images

csv_file = os.path.splitext(filename)[0] + '.csv'
if os.path.exists(csv_file):
    print('corresponding points found')
    cor_points = pd.read_csv(csv_file)
    fix_points = cor_points[['fix_x', 'fix_y']].values
    mov_points = cor_points[['mov_x', 'mov_y']].values
    
    write_pts_file(fix_points, name = 'fix.pts')
    write_pts_file(mov_points, name = 'mov.pts')

print('slice num {}'.format(df.iloc[num].Slice))
print('allen slice {}'.format(allen_slice_num))
print('registering in progress')

# register a slice to allen
trans, moving_spline = register_images(fixed, moving, p, p2)
trans

# transform other images
moving_rigid = transform_image(moving, trans[0])
moving_annot_rigid = annot_revert[transform_image(annot_convert[moving_annot], trans[0], interpolation = False).astype(np.uint32)]
moving_annot_spline = annot_revert[transform_image(annot_convert[moving_annot], trans, interpolation = False).astype(np.uint32)]
moving_borders_spline = transform_image(moving_borders, trans, interpolation = True)

# get some cell positions
if True:
    cells = get_cell_metadata_for_slice_index(df, num, cmd, ccf_pixel_size = ccf_pixel_size, bbox = fix_bbox)
    
    cells_pos_fixed = np.array([cells['fixed_x'], cells['fixed_y']]).T
    write_pts_file(cells_pos_fixed) # write the files to the disk

    # warp the points
    transformixImageFilter = sitk.TransformixImageFilter()
    transformixImageFilter.SetTransformParameterMap(trans)
    transformixImageFilter.SetMovingImage(sitk.GetImageFromArray(moving))
    transformixImageFilter.SetFixedPointSetFileName('points.pts')
    transformixImageFilter.Execute()
    output_points = read_outputpoints_file()
    cells_pos_moving = output_points[:,3]
    cells_pos_moving -= pad_width # remove the pad
    cells_pos_moving += np.array([mov_bbox[1], mov_bbox[0]]) # adjust for moving crop

    zloc = np.ones(len(cells_pos_moving)) * allen_slice_num
    yloc = cells_pos_moving[:,1]
    xloc = cells_pos_moving[:,0]
    
    zloc_int = zloc.astype(int)
    yloc_int = yloc.astype(int)
    xloc_int = xloc.astype(int)
    
    z_valid = ((zloc_int > 0) & (zloc_int < annot.shape[0]))
    y_valid = ((yloc_int > 0) & (yloc_int < annot.shape[1]))
    x_valid = ((xloc_int > 0) & (xloc_int < annot.shape[2]))
    valid = z_valid & y_valid & x_valid

    cells_annotations = annot[zloc_int[valid], yloc_int[valid], xloc_int[valid]]

    ccfx = ccf_pixel_size * zloc[valid] # AP axis
    ccfy = ccf_pixel_size * yloc[valid] #
    ccfz = ccf_pixel_size * xloc[valid] #
    
    index_valid = cells.index[valid]
    cmd.loc[index_valid,'ccfx'] = ccfx
    cmd.loc[index_valid,'ccfy'] = ccfy
    cmd.loc[index_valid,'ccfz'] = ccfz
    cmd.loc[index_valid,'annotation'] = cells_annotations # only assign value annotations
    
# try to get annotations from warped annotated image
if True:
    cells_pos_fixed = np.array([cells['fixed_x'], cells['fixed_y']]).T
    cells_pos_fixed_int = cells_pos_fixed.astype(int)

    cells_pos_fixed_select = (
         (cells_pos_fixed_int[:,1] < (moving_annot_spline.shape[0] - 1)) &
         (cells_pos_fixed_int[:,1] >= 0) &
         (cells_pos_fixed_int[:,0] < (moving_annot_spline.shape[1] - 1)) &
         (cells_pos_fixed_int[:,0] >= 0))
    cells_pos_fixed_int = cells_pos_fixed_int[cells_pos_fixed_select]
    cells_pos_fixed_annotations = moving_annot_spline[cells_pos_fixed_int[:,1],cells_pos_fixed_int[:,0]].astype(int)

    
### This is plotting to compare cells positions to borders ###

color_map = 'subclass_color'

percent_high = 0.95

fig, axs = plt.subplots(nrows = 3, ncols = 4, figsize = (15,10))
fig.suptitle('slice {} to allen {}'.format(df.iloc[num].Slice, allen_slice_num), fontsize=16)

axs[0,0].set_title('fixed slice {}'.format(df.iloc[num].Slice))
axs[0,0].imshow(imagescPercent(fixed,0, percent_high), cmap = 'gray')
#axs[0,0].scatter(cells['fixed_x'], cells['fixed_y'])

axs[0,1].set_title('fixed + rigid')
axs[0,1].imshow(
    imageoverlay(
    imagescPercent(scale_result(fixed),0.0, percent_high),
    imagescPercent(scale_result(moving_rigid),0.0,percent_high)
            ))

axs[0,2].set_title('fixed + spline')
axs[0,2].imshow(
    imageoverlay(
    imagescPercent(scale_result(fixed),0.0, percent_high),
    imagescPercent(scale_result(moving_spline),0.0,percent_high)
            ))

axs[0,3].set_title('fixed + borders spline')
axs[0,3].imshow(imagescPercent(fixed,0, percent_high), cmap = 'gray')
axs[0,3].imshow(border_transparency(moving_borders_spline, RGBval = [1,0,1]))

axs[1,0].set_title('moving allen {}'.format(allen_slice_num))
axs[1,0].imshow(imagescPercent(moving,0, percent_high), cmap = 'gray')

axs[1,1].set_title('moving rigid')
axs[1,1].imshow(imagescPercent(moving_rigid,0, percent_high), cmap = 'gray')

axs[1,2].set_title('moving spline')
axs[1,2].imshow(imagescPercent(moving_spline,0, percent_high), cmap = 'gray')

axs[2,0].set_title('borders')
axs[2,0].set_facecolor((0,0,0))
axs[2,0].imshow(border_transparency(moving_borders, RGBval = [1,0,1]))

axs[2,1].set_title('moving + borders')
axs[2,1].imshow(imagescPercent(moving,0, percent_high), cmap = 'gray')
axs[2,1].imshow(border_transparency(moving_borders, RGBval = [1,0,1]))

axs[2,2].set_title('moving + borders spline')
axs[2,2].imshow(imagescPercent(moving_spline,0, percent_high), cmap = 'gray')
axs[2,2].imshow(border_transparency(moving_borders_spline, RGBval = [1,0,1]))

# all cells
fig3, axs = plt.subplots(1,2, figsize = (15,10))

axs[0].imshow(border_transparency(moving_borders_spline, RGBval = [0,0,0]))

axs[1].set_title('slice {}'.format(slice_to_register))
axs[1].set_facecolor((1,1,1))
axs[1].scatter(cells['fixed_x'], cells['fixed_y'], s = 0.3, c = cells[color_map])
axs[1].imshow(border_transparency(moving_borders_spline, RGBval = [0,0,0]))
print('done')

# Loop through slices

In [ ]:
istart = 1 #4
istop = 14 #58
rescale_percent = 0.99

In [ ]:
# Each run writes to its own iteration directory, preserving the complete visual history.
# Set ALLEN_CCF_ITERATION explicitly for a numbered rerun; default 03 follows the audit iterations.
iteration_number = int(os.environ.get("ALLEN_CCF_ITERATION", "03"))
figpath_allen_ccf = os.path.join(iteration_root, f"iteration_{iteration_number:02d}")
os.makedirs(figpath_allen_ccf, exist_ok=True)
print(f"Registration figures will be written to: {figpath_allen_ccf}")

In [ ]:
pd.read_excel(slice_info_filename)

In [ ]:
matplotlib.use('Agg')
# os.path.join(base_path)

plotQ = True

# Do a loop of a bunch of slices
slices_to_register = np.arange(istart,istop).astype(int)

for slice_to_register in slices_to_register:
    
    print('working on slice {}'.format(slice_to_register))
    
    # Do a big loop to test many registration angles of this slice
    num = slice_to_register - 1

    # df = import_df(slice_info_filename)
    df = pd.read_excel(slice_info_filename)
    
    ccf_pixel_size = 25

    print('{}/{}'.format(slice_to_register,slices_to_register[-1]), end = '\r')

    filename = df.iloc[num].Filename
    slice_id = df.iloc[num].cell_metadata
    allen_slice_num = df.iloc[num].allen_slice_num

    rescale_percent = 0.99

    moving_annot = annot[allen_slice_num]
    moving_borders = borders[allen_slice_num]
    nissl_im = nissl[allen_slice_num]

    # fixed = modify_dapi(df, num, cmd,
    #     cell_types = df.iloc[num].cell_types,
    #     space_modules = df.iloc[num].space_modules,
    #     factor = df.iloc[num].dapi_enhance_factor)

    # moving  = modify_nissl(nissl_im, moving_annot,
    #     annot_dict = df.iloc[num].annots_to_amplify,
    #     factor = df.iloc[num].nissl_enhance_factor)
    
    fixed = modify_dapi(df, num, cmd, cell_types = cell_types_to_amplify, factor = 2)
    moving  = modify_nissl(nissl_im, moving_annot, annot_dict = annots_to_amplify,)

    # make some params
    p, p2 = params_from_df(df, num)
    # crop and pad images
    pad_width = 20
    area_thresh = df.iloc[num].area_thresh
    
    # crop images (may not be necessary since we did it already) and add a pad
    fix_bbox, fixed, _ = crop_and_pad_image(fixed, pad_width = pad_width, area_thresh = area_thresh)
    _, _, moving_borders = crop_and_pad_image(moving, moving_borders, pad_width = pad_width, area_thresh = area_thresh) # be careful here don't overwrite the original moving...
    mov_bbox, moving, moving_annot = crop_and_pad_image(moving, moving_annot, pad_width = pad_width, area_thresh = area_thresh) 
    
    # check if there are corresponding points
    csv_file = os.path.splitext(filename)[0] + '.csv'
    if os.path.exists(csv_file):
        print('corresponding points found')
        cor_points = pd.read_csv(csv_file)
        fix_points = cor_points[['fix_x', 'fix_y']].values
        mov_points = cor_points[['mov_x', 'mov_y']].values

        write_pts_file(fix_points, name = 'fix.pts')
        write_pts_file(mov_points, name = 'mov.pts')
    
    # register a slice to allen
    trans, moving_spline = register_images(fixed, moving, p, p2)

    # transform other images
    moving_rigid = transform_image(moving, trans[0])
    moving_annot_rigid = annot_revert[transform_image(annot_convert[moving_annot], trans[0], interpolation = False).astype(np.uint32)]
    moving_annot_spline = annot_revert[transform_image(annot_convert[moving_annot], trans, interpolation = False).astype(np.uint32)]
    moving_borders_spline = transform_image(moving_borders, trans, interpolation = True)   
    
    # get some cell positions
    cells = get_cell_metadata_for_slice_index(df, num, cmd, ccf_pixel_size = ccf_pixel_size, bbox = fix_bbox)
    
    # only proceed if we have some cell metadata
    if len(cells) > 0:

        cells_pos_fixed = np.array([cells['fixed_x'], cells['fixed_y']]).T #
        write_pts_file(cells_pos_fixed) # write the files to the disk

        # warp the points
        transformixImageFilter = sitk.TransformixImageFilter()
        transformixImageFilter.SetTransformParameterMap(trans)
        transformixImageFilter.SetMovingImage(sitk.GetImageFromArray(moving))
        transformixImageFilter.SetFixedPointSetFileName('points.pts')
        transformixImageFilter.Execute()

        output_points = read_outputpoints_file()

        cells_pos_moving = output_points[:,3]

        cells_pos_moving -= pad_width # remove the pad
        cells_pos_moving += np.array([mov_bbox[1], mov_bbox[0]]) # adjust for moving crop

        zloc = np.ones(len(cells_pos_moving)) * allen_slice_num
        yloc = cells_pos_moving[:,1]
        xloc = cells_pos_moving[:,0]

        # make integer locations
        zloc_int = zloc.astype(int)
        yloc_int = yloc.astype(int)
        xloc_int = xloc.astype(int)

        # take only valid pixels
        z_valid = ((zloc_int > 0) & (zloc_int < annot.shape[0]))
        y_valid = ((yloc_int > 0) & (yloc_int < annot.shape[1]))
        x_valid = ((xloc_int > 0) & (xloc_int < annot.shape[2]))
        valid = z_valid & y_valid & x_valid

        cells_annotations = annot[zloc_int[valid], yloc_int[valid], xloc_int[valid]]

        ccfx = ccf_pixel_size * zloc[valid] # AP axis
        ccfy = ccf_pixel_size * yloc[valid] #
        ccfz = ccf_pixel_size * xloc[valid] #

        index_valid = cells.index[valid] # only use valid index
        cmd.loc[index_valid,'ccfx'] = ccfx
        cmd.loc[index_valid,'ccfy'] = ccfy
        cmd.loc[index_valid,'ccfz'] = ccfz
        cmd.loc[index_valid,'annotation'] = cells_annotations # only assign value annotations
        
        if plotQ:
        
            ### This is plotting to compare cells positions to borders ###

            color_map = 'subclass_color'

            percent_high = 0.99
            fig, axs = plt.subplots(nrows = 3, ncols = 4, figsize = (10,7))
            fig.suptitle('{} slice {} to allen {}'.format(slice_id, df.iloc[num].Slice, allen_slice_num), fontsize=16)

            axs[0,0].set_title('fixed slice {}'.format(df.iloc[num].Slice))
            axs[0,0].imshow(imagescPercent(fixed,0, percent_high), cmap = 'gray')

            axs[0,1].set_title('fixed + rigid')
            axs[0,1].imshow(
                imageoverlay(
                imagescPercent(scale_result(fixed),0.0, percent_high),
                imagescPercent(scale_result(moving_rigid),0.0,percent_high)
                        ))

            axs[0,2].set_title('fixed + spline')
            axs[0,2].imshow(
                imageoverlay(
                imagescPercent(scale_result(fixed),0.0, percent_high),
                imagescPercent(scale_result(moving_spline),0.0,percent_high)
                        ))

            axs[0,3].set_title('fixed + borders spline')
            axs[0,3].imshow(imagescPercent(fixed,0, percent_high), cmap = 'gray')
            axs[0,3].imshow(border_transparency(moving_borders_spline, RGBval = [1,0,1]))

            axs[1,0].set_title('moving allen {}'.format(allen_slice_num))
            axs[1,0].imshow(imagescPercent(moving,0, percent_high), cmap = 'gray')

            axs[1,1].set_title('moving rigid')
            axs[1,1].imshow(imagescPercent(moving_rigid,0, percent_high), cmap = 'gray')

            axs[1,2].set_title('moving spline')
            axs[1,2].imshow(imagescPercent(moving_spline,0, percent_high), cmap = 'gray')


            axs[2,0].set_title('borders')
            axs[2,0].set_facecolor((0,0,0))
            axs[2,0].imshow(border_transparency(moving_borders, RGBval = [1,0,1]))

            axs[2,1].set_title('moving + borders')
            axs[2,1].imshow(imagescPercent(moving,0, percent_high), cmap = 'gray')
            axs[2,1].imshow(border_transparency(moving_borders, RGBval = [1,0,1]))

            axs[2,2].set_title('moving + borders spline')
            axs[2,2].imshow(imagescPercent(moving_spline,0, percent_high), cmap = 'gray')
            axs[2,2].imshow(border_transparency(moving_borders_spline, RGBval = [1,0,1]))
            
            # all cells
            fig3, axs = plt.subplots(1,2, figsize = (15,10))

            axs[0].imshow(border_transparency(moving_borders_spline, RGBval = [0,0,0]))

            axs[1].set_title('borders')
            axs[1].set_facecolor((1,1,1))
            axs[1].scatter(cells['fixed_x'], cells['fixed_y'], s = 0.3, c = cells[color_map])
            axs[1].imshow(border_transparency(moving_borders_spline, RGBval = [0,0,0]))

            fig.tight_layout(pad=2.0)
            fig.subplots_adjust(top=0.9)
            
            fig_path1 = os.path.join(figpath_allen_ccf, 
                '{}_slice_{}_allen_{}_reg_{}.jpg'.format(slice_id, str(df.iloc[num].Slice).zfill(3), str(allen_slice_num).zfill(3), num))
            fig_path2 = os.path.join(figpath_allen_ccf, 
                '{}_slice_{}_allen_{}_reg_{}_cells_all.jpg'.format(slice_id, str(df.iloc[num].Slice).zfill(3), str(allen_slice_num).zfill(3), num))

            # fig_path1 = os.path.join(figpath_allen_ccf, 
            #                          '{slice_id}_slice_{}_allen_{}_reg_{}.jpg'.format(str(df.iloc[num].Slice).zfill(3), str(allen_slice_num).zfill(3), num))
            # fig_path2 = os.path.join(figpath_allen_ccf, 
            #                          'slice_{}_allen_{}_reg_{}_cells_all.jpg'.format(str(df.iloc[num].Slice).zfill(3), str(allen_slice_num).zfill(3), num))

            fig.savefig(fig_path1, dpi = 600)
            fig3.savefig(fig_path2, dpi = 600)
            plt.close('all')
print('done')    

In [ ]:
cmd
figpath_allen_ccf

In [ ]:
# save the 2d registration result
ccf2d_results_path = os.path.join(analysis_reg_path, 'ccf2d.csv')
cmd.to_csv(ccf2d_results_path, index = True)

# 3D Registration

In [ ]:
ccf2d_results_path = os.path.join(analysis_reg_path, 'ccf2d.csv')

if True: # start from this step...
    cmd = pd.read_csv(ccf2d_results_path)
    cmd.set_index('cell_id', inplace = True)

In [ ]:
cmd

In [ ]:
for c in cmd.columns:
    print('column {} number of NAs: {}'.format(c,cmd[c].isna().sum()))

In [ ]:
# # remove the bad slice
# for bad_slice in bad_slices:
#     cmd = cmd[cmd.slice_id != bad_slice]


In [ ]:
cmd.shape

In [ ]:
cmd[~cmd.ccfx.isna()].shape

In [ ]:
# take only valid pixel positions in the annotated image
cmd = cmd[~cmd.ccfx.isna()]

ccfx_valid = (cmd.ccfx/ccf_pixel_size).astype(int) < annot.shape[0]
ccfy_valid = (cmd.ccfy/ccf_pixel_size).astype(int) < annot.shape[1]
ccfz_valid = (cmd.ccfz/ccf_pixel_size).astype(int) < annot.shape[2]

ccf_valid = (ccfx_valid & ccfy_valid & ccfz_valid)

cmd = cmd[ccf_valid]
cmd


In [ ]:
# make annotations ints
cmd['annotation'] = cmd['annotation'].values.astype(int)

In [ ]:
cmd['annotation'].unique()

### cell type to annotations

In [ ]:
structure_df_file

In [ ]:
# this is the dataframe that has the cell type to annotated region associations
#structure_df = pd.read_excel(r'cell_types_to_structures_v1.xlsx', header=0)

structure_df = pd.read_excel(structure_df_file, header=0)
columns_to_parse = ['cell_types','annotations']
for col in columns_to_parse:
    structure_df[col] = structure_df[col].fillna('[]')
    structure_df[col] = structure_df[col].apply(lambda x: ast.literal_eval(str(x)))

structure_df['use'] = structure_df['use'].fillna('')

structure_df['start'] = structure_df['start'].fillna(0)
structure_df['start'] = structure_df['start'].astype(int)
structure_df['stop'] = structure_df['stop'].fillna(len(annot))
structure_df['stop'] = structure_df['stop'].astype(int)
structure_df

In [ ]:
cell_types_all = cmd.subclass_label_transfer.unique()
cell_types = [item for sublist in structure_df.cell_types for item in sublist]
for c in cell_types:
    if c in cell_types_all:
        pass #print('{} exists'.format(c))
    elif c.startswith('SM_'):
        pass
    else:
        print('{} does not exist'.format(c))

In [ ]:
# assign an intensity to the cell types in 16 bit space

structure_df['intensity'] = 0
factor = 500

num_regions = np.sum(structure_df.use != '')
intensities = factor * (np.arange(num_regions) + 2) # leave space for an 
# why plus 2? we don't want a zero value and we want to leave space for an 'alls cells' label

use_sequential = False
if use_sequential:
    counter = 1
    for i,row in structure_df.iterrows():
        if row.use:
            structure_df.loc[i, 'intensity'] = counter * factor
            counter += 1

use_random = True
if use_random:
    np.random.seed(12)
    np.random.shuffle(intensities)
    counter = 0
    for i,row in structure_df.iterrows():
        if row.use:
            structure_df.loc[i, 'intensity'] = intensities[counter]
            counter += 1

structure_df[structure_df.use != ''][['cell_types','intensity']].sort_values('intensity')

In [ ]:
intensity_unique = np.unique(structure_df.intensity)
intensity_unique[0] = factor # base value
intensity_unique = np.insert(intensity_unique, 0, 0) # do this to deal with the zero value


intensity_convert = np.zeros(intensity_unique[-1] + 1, dtype=np.uint16)
for i,val in enumerate(intensity_unique):
    intensity_convert[val] = i

intensity_revert = np.zeros(len(intensity_unique), dtype=np.uint16)
for i,val in enumerate(intensity_unique):
    intensity_revert[i] = val

### Make the 3D moving Image using Selected Annotations

In [ ]:
# make the pseudo color moving image
# take only the regions of the annotation that are in the structure df

annot_select = np.zeros(annot.shape, dtype = np.uint16)

# first make a low intensity bin for all the cells
annot_select[annot > 0] = 1 * factor

# next add space modules
for i, row in structure_df.iterrows():
    if row.use == 'space_modules':
        mask = np.isin(annot, row.annotations)
        
        #start stop condition
        mask[:row.start] = False
        mask[row.stop:] = False
        annot_select[mask] = row.intensity
        

# next add cell types
for i, row in structure_df.iterrows():
    if row.use == 'cell_types':
        mask = np.isin(annot, row.annotations)
        
        # do a special case for Ependymal cells
        # this will make hollow ventricals
        if row.cell_types[0] == 'Ependymal NN':
            mask_erode = scipy.ndimage.binary_erosion(mask, iterations = 2)
            ventricle_mask = np.logical_and(mask, np.logical_not(mask_erode))   
            mask = ventricle_mask
        
        #start stop condition
        mask[:row.start] = False
        mask[row.stop:] = False
        
        annot_select[mask] = row.intensity
        
print(np.unique(annot_select))

#add a surface layer of cells to mimic the VLMC cells:
size = -1 # expand or shrink the annotation for vlmc cells?
thickness = 2 # then dilate to thickness
start_frame = 71 # dont use VLMC before this point

if size == 1:
    mask1 = annot > 0
    mask2 = scipy.ndimage.binary_dilation(mask1, iterations = thickness)
if size > 0:
    mask1 = scipy.ndimage.binary_dilation(annot > 0, iterations = size)
    mask2 = scipy.ndimage.binary_dilation(mask1, iterations = thickness)
if size < 0:
    size = np.abs(size)
    mask1 = scipy.ndimage.binary_erosion(annot > 0, iterations = size)
    mask2 = scipy.ndimage.binary_dilation(mask1, iterations = thickness)

surface = np.logical_and(mask2, np.logical_not(mask1))
surface[:start_frame] = False
    
### add the midline intersection with cortex
mask_ctx = np.isin(annot, allen_name_to_annots['Cerebral cortex'])
mask_midline = np.zeros(annot.shape, dtype = bool)
mask_midline[:,:,midline-1:midline + 2] = True
mask_midline = np.logical_and(mask_ctx, mask_midline)

surface = np.logical_or(mask_midline, surface)


# get the intensity value
for i, row in structure_df.iterrows():
    if 'VLMC NN' in row.cell_types:
        print('vlmc surface added')
        annot_select[surface] = row.intensity
        #annot_select[midline] = row.intensity # add midline here?
        
        
skimage.io.imsave('selected_annotations.tif', annot_select)

### Make the 3D fixed image using cell type positions

In [ ]:
# make a pseudo color fixed image using the ccf location and the intensity above

arr = np.zeros(annot.shape, dtype = np.uint16)

# # first add space module
# for i, row in structure_df.iterrows():
#     if row.use == 'space_modules':
#         mask = np.zeros(annot.shape, bool)
#         cmd_temp = cmd[cmd.spatial_modules_level_1_name.isin(row.cell_types)].dropna()
#         i = (np.rint(cmd_temp.ccfx.values)/ccf_pixel_size).astype(int)
#         j = (cmd_temp.ccfy.values/ccf_pixel_size).astype(int)
#         k = (cmd_temp.ccfz.values/ccf_pixel_size).astype(int)
#         mask[i,j,k] = True
#         mask[:row.start] = False
#         mask[row.stop:] = False
#         arr[mask] = row.intensity
        
# next add cell types
for i, row in structure_df.iterrows():        
    if row.use == 'cell_types':
        mask = np.zeros(annot.shape, bool)
        cmd_temp = cmd[cmd.subclass_label_transfer.isin(row.cell_types)].dropna()
        i = (np.rint(cmd_temp.ccfx.values)/ccf_pixel_size).astype(int)
        j = (cmd_temp.ccfy.values/ccf_pixel_size).astype(int)
        k = (cmd_temp.ccfz.values/ccf_pixel_size).astype(int)
            
        # make a special case for VLMC to add surface cells
        # first make a mask
        
        if 'VLMC NN' in row.cell_types:
            dilate_1 = 1
            dilate_2 = 12
            vlmc_mask = np.logical_xor(annot > 0, scipy.ndimage.binary_dilation(annot > 0, iterations = dilate_1))
            vlmc_mask[:,:,-1] = True
            vlmc_mask = scipy.ndimage.binary_dilation(vlmc_mask, iterations = dilate_2)
            arr_vlmc_only = np.zeros(annot.shape, dtype = np.uint16)
            arr_vlmc_only[i,j,k] = row.intensity
            arr_vlmc_only = arr_vlmc_only * vlmc_mask
            arr[arr_vlmc_only > 0] = row.intensity # safer than adding...
        else:
            mask[i,j,k] = True
            mask[:row.start] = False
            mask[row.stop:] = False
            arr[mask] = row.intensity

# try a different way off expanding labels
arr2 = skimage.segmentation.expand_labels(arr, distance = 1)

# set any pixel that has a cell to a low intensity value
# but don't overwrite any of our expanded pixels
cmd_temp = cmd.dropna()
used_cell_types = list(structure_df[structure_df.use == 'cell_types'].cell_types.values)
used_cell_types = [item for sublist in used_cell_types for item in sublist]
cmd_temp = cmd_temp[~cmd_temp.subclass_label_transfer.isin(used_cell_types)]
arr_temp = np.zeros(arr.shape, dtype = bool)
i = (np.rint(cmd_temp.ccfx.values)/ccf_pixel_size).astype(int)
j = (cmd_temp.ccfy.values/ccf_pixel_size).astype(int)
k = (cmd_temp.ccfz.values/ccf_pixel_size).astype(int)
arr_temp[i,j,k] = True
arr_temp = skimage.morphology.binary_dilation(arr_temp, skimage.morphology.ball(1))

mask = np.logical_and(arr_temp, np.logical_not(arr2 > 0))

arr2[mask] = 1 * factor

In [ ]:
# rgb_lut = RGB_lut_255(len(np.unique(annot_select)))
n_colors = int(max(arr.max(), arr2.max(), annot_select.max()) / factor) + 1
rgb_lut = RGB_lut_255(n_colors)
skimage.io.imsave('selected_cells_RGB.tif', rgb_lut[(arr/factor).astype(np.uint8)])
skimage.io.imsave('selected_cells_dilate_RGB.tif', rgb_lut[(arr2/factor).astype(np.uint8)])
skimage.io.imsave('selected_annotations_RGB.tif', rgb_lut[(annot_select/factor).astype(np.uint8)])

In [ ]:
#include the 3D nissl image

output = np.copy(nissl)
mean_val = np.mean(output[output > 0])
factor = 2

# make a surface vlmc layer
mask = annot > 0
mask = skimage.morphology.binary_erosion(mask,skimage.morphology.ball(1))
surface = np.logical_xor(mask, skimage.morphology.binary_dilation(mask, skimage.morphology.ball(2)))    
output = output * mask # clear outside
output += surface * mean_val * factor # add vlmc layer

# Ependymal NN ventricles
annots_ENN = [81,129]
ventrical_mask = np.isin(annot, annots_ENN)
edge_mask = np.logical_xor(ventrical_mask, skimage.morphology.binary_erosion(ventrical_mask, skimage.morphology.ball(1)))

output = output * np.logical_not(ventrical_mask).astype(int) # remove ventricals
output += edge_mask * mean_val * factor # add surface

# msn
mask_mimic = (annot == 754) # bottom part of msn d1 gaba
output[mask_mimic] = mean_val * factor

pseudo_nissl = np.copy(output).astype(np.uint16)

skimage.io.imsave('pseudo_nissl.tif', pseudo_nissl)

In [ ]:
# make a 3d dapi-like image using cell positions

pseudo_dapi = np.zeros(annot.shape, dtype = np.uint16)

ccfx_valid = (cmd.ccfx/ccf_pixel_size).astype(int) < (annot.shape[2] - 1)
ccfy_valid = (cmd.ccfy/ccf_pixel_size).astype(int) < (annot.shape[1] - 1)
ccfz_valid = (cmd.ccfz/ccf_pixel_size).astype(int) < (annot.shape[0] - 1)

ccf_valid = (ccfx_valid & ccfy_valid & ccfz_valid)

cmd_temp = cmd[ccf_valid]

cmd_temp = cmd.dropna()
i = np.rint(cmd_temp.ccfx.values/ccf_pixel_size).astype(int)
j = (cmd_temp.ccfy.values/ccf_pixel_size).astype(int)
k = (cmd_temp.ccfz.values/ccf_pixel_size).astype(int)
pseudo_dapi[i,j,k] += 1

# enhance the cells we used above also!
enhance_cells = ['DG Glut', 'CA1-ProS Glut', 'CA3 Glut', 'VLMC NN', 'ABC NN', 'Astroependymal NN', 'Ependymal NN', 'CHOR NN']

cmd_temp = cmd[cmd.subclass_label_transfer.isin(enhance_cells)]
i = np.rint(cmd_temp.ccfx.values/ccf_pixel_size).astype(int)
j = (cmd_temp.ccfy.values/ccf_pixel_size).astype(int)
k = (cmd_temp.ccfz.values/ccf_pixel_size).astype(int)
pseudo_dapi[i,j,k] += 3

pseudo_dapi *= 5000

pseudo_dapi = skimage.filters.gaussian(pseudo_dapi, [1.5,1,1], preserve_range=True).astype(np.uint16)

# special case to mimic nissl
pseudo_dapi[mask_mimic] = np.mean(pseudo_dapi[pseudo_dapi > 0]) * 3

In [ ]:
# make some spline params
num_histogram_bins = 64

p2 = sitk.GetDefaultParameterMap("bspline")
p2['NumberOfSpatialSamples'] = L2P([10000])

p2['NumberOfHistogramBins'] = L2P([num_histogram_bins])
p2['MaximumNumberOfIterations'] = L2P([1000])
p2['FinalGridSpacingInPhysicalUnits'] = L2P([])

p2['FixedImagePyramid'] = L2P(['FixedSmoothingImagePyramid','FixedSmoothingImagePyramid']) ####
p2['MovingImagePyramid'] = L2P(['MovingSmoothingImagePyramid','MovingSmoothingImagePyramid']) ####

p2['ImageSampler'] = L2P(['RandomCoordinate','RandomCoordinate'])
p2['Interpolator'] = L2P(['BSplineInterpolator','BSplineInterpolator'])

p2['NumberOfResolutions'] = L2P([3])
p2['GridSpacingSchedule'] = L2P([4,3,2])
p2['FinalGridSpacingInVoxels'] =  L2P([30,30,30])

p2['WriteResultImageAfterEachResolution'] = L2P(['true'])
p2['WriteTransformParametersEachResolution'] = L2P(['true'])

p2['NewSamplesEveryIteration'] = L2P(['true']) # useful?
p2['MaximumStepLength'] = L2P([1])

p2['Metric'] = L2P(['AdvancedMattesMutualInformation','AdvancedNormalizedCorrelation'])# ,'TransformBendingEnergyPenalty'])

p2['Metric0Weight'] = L2P([0.2])
p2['Metric1Weight'] = L2P([0.8])


print('number sampling points / total pixels : {}/{}'.format(int(p2['NumberOfSpatialSamples'][0]), np.prod(arr.shape)))
print('sampling ratio : {}'.format(int(p2['NumberOfSpatialSamples'][0])/np.prod(arr.shape)))

p2.asdict()

In [ ]:
elastixImageFilter = sitk.ElastixImageFilter()

elastixImageFilter.SetParameterMap(p2)

fixed_1 = np.pad(pseudo_dapi, ((0,0),(0,0),(0,10)))
fixed_2 = np.pad(arr2, ((0,0),(0,0),(0,10)))

moving_1 = np.pad(pseudo_nissl, ((0,0),(0,0),(0,10)))
moving_2 = np.pad(annot_select, ((0,0),(0,0),(0,10)))

# clear all data before the first and after the last frame of actual data
start_frame = 19 # first frame where the data is
stop_frame = 451 # last frame where there is data
# this helps the 3d registration not get squashed

mask = np.zeros(fixed_1.shape, bool)
mask[start_frame:stop_frame] = True
mask = np.logical_not(mask)

fixed_1[mask] = 0
fixed_2[mask] = 0
moving_1[mask] = 0
moving_2[mask] = 0

skimage.io.imsave('fixed_1.tif', fixed_1)
skimage.io.imsave('fixed_2.tif', fixed_2)
skimage.io.imsave('moving_1.tif', moving_1)
skimage.io.imsave('moving_2.tif', moving_2)

elastixImageFilter.AddFixedImage(sitk.GetImageFromArray(fixed_1))
elastixImageFilter.AddFixedImage(sitk.GetImageFromArray(fixed_2))

elastixImageFilter.AddMovingImage(sitk.GetImageFromArray(moving_1))
elastixImageFilter.AddMovingImage(sitk.GetImageFromArray(moving_2))

elastixImageFilter.Execute()

result_image = sitk.GetArrayFromImage(elastixImageFilter.GetResultImage())
trans = elastixImageFilter.GetTransformParameterMap()

t = int(time.time())

In [ ]:
# use the convert revert trick to fix the interpolation issue
# save the parameter file so it can be used later

moving_spline_1 = transform_image(moving_1, trans, interpolation = False).astype(np.int16)
moving_spline_2 = intensity_revert[transform_image(intensity_convert[moving_2], trans, interpolation = False).astype(np.uint16)]

sitk.WriteParameterFile(p2, 'Params_3D_2chan_{}.txt'.format(t))
skimage.io.imsave('moving_3D_spline_chan1_{}.tif'.format(t), moving_spline_1.astype(np.uint16))
skimage.io.imsave('moving_3D_spline_chan2_{}.tif'.format(t), moving_spline_2.astype(np.uint16))

In [ ]:
# transform the 25 um annotated image

annot_pad = np.pad(annot, ((0,0),(0,0),(0,10)))

spline_25 = transform_image(annot_convert[annot_pad], trans, interpolation = False)
spline_25 = annot_revert[spline_25.astype(np.uint32)]

sitk.WriteParameterFile(trans[0], 'transformation_25.txt')
skimage.io.imsave('moving_3D_spline_25_{}.tif'.format(t), spline_25)

In [ ]:
# also transform the 10 um annotated image

d = (np.array(moving_spline_2.shape) * 2.5).astype(int)
print(d)

annot_10, header = nrrd.read(ara_annot_10_file, index_order='F')
annot_10 = annot_10[:d[0],:d[1],:d[2]]
annot_10.shape

In [ ]:
trans_10 = copy.copy(trans)

spacing = [0.4,0.4,0.4]

trans_10[0]['Size'] = L2P(np.flip(d))
trans_10[0]['Spacing'] = L2P(spacing)

spline_10 = transform_image(annot_convert[annot_10], trans_10, interpolation = False, spacing = tuple(spacing))
spline_10 = annot_revert[spline_10.astype(np.uint32)]

print(spline_10.dtype)
print(spline_10.shape)

sitk.WriteParameterFile(trans_10[0], 'transformation_10um.txt')
skimage.io.imsave('moving_3D_spline_10_{}.tif'.format(t), spline_10)

### Transform the cell coordinates

In [ ]:
%matplotlib inline

In [ ]:
# read in cell metadata that has ccf coordinates from reconstruction
ccf2d_results_path = os.path.join(analysis_reg_path, 'ccf2d.csv')
cmd = pd.read_csv(ccf2d_results_path)
cmd.set_index('cell_id', inplace = True)
len(cmd)

In [ ]:
# # show where the NAs are
# slice_id = 'co1_slice30'
# test_slice = cmd[cmd['slice_id'] == slice_id]
# test_slice_not_NA = test_slice[~test_slice.ccfx.isna()]
# test_slice_NA = test_slice[test_slice.ccfx.isna()]

# plt.scatter(test_slice_not_NA.center_x, test_slice_not_NA.center_y, color = 'b')
# plt.scatter(test_slice_NA.center_x, test_slice_NA.center_y, color = 'r')

In [ ]:
# be careful of this drop NA step 
# it is important for transformix to not have any NA values

cmd = cmd[~cmd.annotation.isna()]
len(cmd)

In [ ]:
if np.any(cmd.columns.isin(['ccfx_2', 'ccfy_2', 'ccfz_2', 'annotation_2'])):
    cmd.drop(columns=['ccfx_2', 'ccfy_2', 'ccfz_2', 'annotation_2'], inplace = True)

z = cmd.ccfx.values/ccf_pixel_size
y = cmd.ccfy.values/ccf_pixel_size
x = cmd.ccfz.values/ccf_pixel_size
cells_pos = np.array([x, y, z]).T # - 0.5 # correction to ITK Voroni pixel?

# write the files to the disk
write_pts_file(cells_pos)

In [ ]:
# warp the points
transformixImageFilter = sitk.TransformixImageFilter()
transformixImageFilter.SetTransformParameterMap(trans)
transformixImageFilter.SetMovingImage(sitk.GetImageFromArray(moving_1))
transformixImageFilter.SetFixedPointSetFileName('points.pts')
transformixImageFilter.Execute()

In [ ]:
# read back the transformed points
output_points = read_outputpoints_file()
cells_pos_moving = output_points[:,3]

In [ ]:
xloc = cells_pos_moving[:,0]
yloc = cells_pos_moving[:,1]
zloc = cells_pos_moving[:,2]

cells_pos_select = ((zloc.astype(int) < annot.shape[0] - 1) & (zloc.astype(int) >= 0) &
                    (yloc.astype(int) < annot.shape[1] - 1) & (yloc.astype(int) >= 0) &
                    (xloc.astype(int) < annot.shape[2] - 1) & (xloc.astype(int) >= 0))

xloc = xloc[cells_pos_select]
yloc = yloc[cells_pos_select]
zloc = zloc[cells_pos_select]

cells_annotations = annot[zloc.astype(int), yloc.astype(int), xloc.astype(int)]

idx = cmd.index[cells_pos_select]
cmd.loc[idx, 'ccfx_2'] = zloc * ccf_pixel_size
cmd.loc[idx, 'ccfy_2'] = yloc * ccf_pixel_size
cmd.loc[idx, 'ccfz_2'] = xloc * ccf_pixel_size
cmd.loc[:, 'annotation_2'] = np.nan # be careful here assigning NAN as default value.. is this smart?
cmd.loc[idx, 'annotation_2'] = cells_annotations

In [ ]:
# save the 3d registration result
ccf3d_results_path = os.path.join(analysis_reg_path, 'ccf3d.csv')
cmd.to_csv(ccf3d_results_path, index = True)

# cmd.to_csv('wb3_co1_all_ccf3d.csv', index = True)

### display registration results

In [ ]:
annot_10, header = nrrd.read(ara_annot_10_file, index_order='F')
print(annot.shape)

In [ ]:
# this is for a curved boundary from 3D registration

def get_curved_border(cmd_subset, border_image3d, pixsize = 10, expand = 20, reflect = True):
    cmd_subset = cmd_subset.dropna(subset=['ccfx_2', 'ccfy_2', 'ccfz_2'])
    imdim = border_image3d[0].shape
    z_contour = np.zeros(imdim, dtype = np.uint16)
    y = (cmd_subset.ccfy_2/pixsize).astype(int)
    x = (cmd_subset.ccfz_2/pixsize).astype(int)
    z_contour[y,x] = (cmd_subset.ccfx_2/pixsize).astype(int)
    z_contour = skimage.segmentation.expand_labels(z_contour, expand)
    yy, xx = np.mgrid[0:imdim[0], 0:imdim[1]]
    output = border_image3d[z_contour, yy, xx]
    if reflect:
        center = int(imdim[1]/2)
        output[:,center:] = output[:,center:0:-1]
    return output

# this is for a 2D registration

def get_2D_border(cmd_subset, border_image3d, pixsize = 10, expand = 20, reflect = True):
    cmd_subset = cmd_subset.dropna(subset=['ccfx', 'ccfy', 'ccfz'])
    imdim = border_image3d[0].shape
    z_contour = np.zeros(imdim, dtype = np.uint16)
    y = (cmd_subset.ccfy/pixsize).astype(int)
    x = (cmd_subset.ccfz/pixsize).astype(int)
    z_contour[y,x] = (cmd_subset.ccfx/pixsize).astype(int)
    z_contour = skimage.segmentation.expand_labels(z_contour, expand)
    yy, xx = np.mgrid[0:imdim[0], 0:imdim[1]]
    output = border_image3d[z_contour, yy, xx]
    if reflect:
        center = int(imdim[1]/2)
        output[:,center:] = output[:,center:0:-1]
    return output

# Use the actual Allen volume sampled at each cell's AP coordinate.
# The original diagnostic expanded sparse cell-assigned labels, producing density-dependent
# Voronoi-like contours that could look misregistered even when the image registration was good.
def get_cell_annotation_border_2D(cmd_subset, annotation_image3d, pixsize=10, expand=30):
    return get_2D_border(
        cmd_subset, annotation_image3d, pixsize=pixsize, expand=expand, reflect=False
    )

# add boundaries between regions
# added gaussian smoothing (set to zero for no smoothing)
def make_vector_outlines(im, smoothing = 0, threshhold = 0.5):
    output = []
    vals = np.unique(im)[1::] # this one drops the zero value
    for v in vals:
        mask = (im == v).astype(float)
        if smoothing == 0:
            mask_contour = skimage.measure.find_contours(mask, 0.99)
        else:
            mask = skimage.filters.gaussian(mask, sigma = smoothing)
            mask_contour = skimage.measure.find_contours(mask, threshhold)
        output.append(mask_contour)
    output = [item for sublist in output for item in sublist]
    return output

# Use the actual Allen volume sampled along the reconstructed curved AP surface.
def get_cell_annotation_border(cmd_subset, annotation_image3d, pixsize=10, expand=30):
    return get_curved_border(
        cmd_subset, annotation_image3d, pixsize=pixsize, expand=expand, reflect=False
    )


In [ ]:
cmd_temp = cmd[cmd.sample_id == "roi4_run3"]

# Use cell-level annotations when available (better borders); else curved surface from volume
curved = get_cell_annotation_border(cmd_temp, annot_10, pixsize=10, expand=30)
outlines = make_vector_outlines(curved, smoothing=0)

fig = plt.figure()
# Use exact hex codes from subclass_color
colors = matplotlib.colors.to_rgba_array(cmd_temp.subclass_color.tolist())
plt.scatter(cmd_temp.ccfz_2/10, cmd_temp.ccfy_2/10, s=1, c=colors)
for outline in outlines:
    plt.plot(*np.flip(outline.T), 'k', linewidth=1)
ax = plt.gca()
ax.set_ylim(0, 800)
ax.set_xlim(0, 1140)
ax.invert_yaxis()


In [ ]:
borders_2d = get_cell_annotation_border_2D(cmd_temp, annot_10, pixsize=10, expand=30)
outlines = make_vector_outlines(borders_2d, smoothing=0)

fig = plt.figure()
# Use exact hex codes from lvl2_colors
colors = matplotlib.colors.to_rgba_array(cmd_temp.subclass_color.tolist())
plt.scatter(cmd_temp.ccfz/10, cmd_temp.ccfy/10, s=.5, c=colors)
for outline in outlines:
    plt.plot(*np.flip(outline.T), 'k', linewidth=1)
ax = plt.gca()
ax.set_ylim(0, 800)
ax.set_xlim(0, 1140)
ax.invert_yaxis()
ax.set_aspect('equal')
# plt.xticks([])
# plt.yticks([])
plt.axis('off')
plt.tight_layout()
fig.savefig('test.png', transparent=True, dpi=300)


In [ ]:
# assemble and order all the slice ids
slice_ids = cmd.sample_id.unique()
slice_ids

In [ ]:
transparent = False

plt.ioff()

for sid in slice_ids:
    print('working on slice id {}'.format(sid), end = '\r')
    cmd_temp = cmd[cmd.sample_id == sid]
    curved = get_cell_annotation_border(cmd_temp, annot_10, pixsize=10, expand=30)
    outlines = make_vector_outlines(curved, smoothing=0)
    
    fig = plt.figure(figsize = [7,10])
    plt.scatter(cmd_temp.ccfz_2/10, cmd_temp.ccfy_2/10, s = .5, c = cmd_temp.subclass_color)
    for outline in outlines:
        plt.plot(*np.flip(outline.T), 'k', linewidth = 1)
    ax = plt.gca()
    ax.set_title(f'{sid}: true Allen 10 µm contours (plotting-only iteration 03)')
    ax.set_ylim(0,800)
    ax.set_xlim(0,1140)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    plt.axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(figpath_allen_ccf, '{} 3d reg.png'.format(sid)), transparent = transparent, dpi = 300)
    plt.close()
plt.ion()

In [ ]:
transparent = False

plt.ioff()

for sid in slice_ids:
    print('working on slice id {}'.format(sid), end = '\r')
    cmd_temp = cmd[cmd.sample_id == sid]
    curved = get_cell_annotation_border_2D(cmd_temp, annot_10, pixsize=10, expand=30)
    outlines = make_vector_outlines(curved, smoothing = 0)
    
    fig = plt.figure(figsize = [7,10])
    plt.scatter(cmd_temp.ccfz/10, cmd_temp.ccfy/10, s = .5, c = cmd_temp.subclass_color)
    for outline in outlines:
        plt.plot(*np.flip(outline.T), 'k', linewidth = 1)
    ax = plt.gca()
    ax.set_title(f'{sid}: true Allen 10 µm contours (plotting-only iteration 03)')
    ax.set_ylim(0,800)
    ax.set_xlim(0,1140)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    plt.axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(figpath_allen_ccf, '{} 2d reg.png'.format(sid)), transparent = transparent, dpi = 300)
    plt.close()

plt.ion()

In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""# Registration optimization audit and iteration library

## Outcome

All 13 available registrations were reviewed independently. The original 2D DAPI–Allen panels already show strong agreement at the outer tissue boundary, cortical laminae, hippocampus, ventricles, midline, and ventral structures. The duplicated workbook therefore retains the original registration parameters: changing `allen_slice_num`, `rot_init`, `scale_x`, `scale_y`, or spline flexibility without evidence of a coherent error would risk degrading valid registrations.

The apparent problems in the original `*3d reg.png` files were primarily diagnostic-code defects:

1. Contours were generated by expanding sparse cell-assigned annotation labels, creating density-dependent/Voronoi-like boundaries rather than true Allen boundaries.
2. The plotting loop selected `cmd.slice_id`, although the saved table uses `sample_id`.
3. `dropna()` considered every metadata column, silently removing valid coordinate rows and producing missing contours.

The corrected workflow samples the true 10 µm Allen annotation volume along each slice's AP surface, filters missing values only on coordinate columns, and uses `sample_id`. The stored 3D transform was also verified to be an identity transform (no coordinate changed), so the final figures audit the already-strong 2D registrations rather than claiming a non-existent global deformation.

## Parameter decision guide

- `allen_slice_num`: first parameter to change for a coherent AP morphology mismatch (hippocampal/ventricular shape differs despite otherwise good registration).
- `rot_init` / `reflect`: only for gross orientation errors; all current orientations are correct.
- `scale_x`, `scale_y`: use for systematic anisotropic size mismatch before affine/spline registration; no current slice shows a consistent scale bias.
- `right_crop`: use only when contralateral tissue or background signal contaminates the fixed image.
- `spline_grid_size`: increase for smoother/less flexible warps; decrease only when a real local anatomical deformation remains after affine registration.
- `iterations` and `histogram_bins`: convergence/intensity-metric controls, not remedies for wrong AP level or tissue tears.
- `area_thresh`: affects tissue cropping and should change only when the crop excludes true tissue or includes debris.
- `cor_pts_weight`: only active when a matching landmark CSV exists beside the DAPI TIFF.

## Iteration record

| Iteration | Change | Observed impact | Decision |
|---:|---|---|---|
| 0 | Isolated copy of baseline outputs and tables | Several contours looked inconsistent despite excellent image-to-image registration panels | Audit contour-generation code before changing registration parameters |
| 1 | Sample true atlas labels and use `sample_id` | Point clouds appeared but contours were absent | Found broad `dropna()` removed rows due to unrelated missing metadata |
| 2 | Restrict null filtering to coordinate columns | True Allen contours align with major landmarks in all 13 slices; containment 93.7–96.7% | Stop: registrations are already strong; retain baseline Excel parameters |

## Final per-slice assessment

Scores are the fraction of valid cell coordinates inside non-background Allen tissue, expressed on a 10-point scale and interpreted together with landmark agreement.

| Sample | Slice → Allen | Score / 10 | Qualitative assessment | Parameter decision |
|---|---:|---:|---|---|
| roi1_run1 | 1 → 294 | 9.37 | Good cortex/hippocampus agreement; mild dorsal/lateral point spill | Retain; `allen_slice_num` then `scale_x/y` would be the diagnostic order only if source anatomy contradicts this |
| roi1_run2 | 2 → 283 | 9.65 | Excellent outer boundary, laminae, hippocampus, and ventral fit | No change |
| roi2_run1 | 3 → 280 | 9.65 | Excellent global and internal landmark agreement | No change |
| roi2_run2 | 4 → 275 | 9.60 | Excellent; slight nonsystematic peripheral spill | No change |
| roi2_run4 | 13 → 282 | 9.66 | Excellent cortex/hippocampus and ventral fit | No change |
| roi3_run1 | 5 → 317 | 9.66 | Good posterior fit; medial gaps follow tissue loss/cut geometry | No change; do not warp atlas into missing tissue |
| roi3_run2 | 6 → 308 | 9.66 | Good posterior morphology; retained landmarks align around tissue discontinuities | No change |
| roi3_run3 | 12 → 279 | 9.67 | Excellent cortex, hippocampus, striatum, and ventral fit | No change |
| roi3_run4 | 11 → 269 | 9.62 | Excellent overall; minor local ventral/medial outliers | No change |
| roi4_run1 | 7 → 285 | 9.67 | Excellent global shape and landmark fit | No change |
| roi4_run2 | 8 → 287 | 9.54 | Very good; mild ventrolateral spill and small tissue tears | No change; consider `right_crop` only if source-image contamination is confirmed |
| roi4_run3 | 10 → 288 | 9.65 | Excellent cortex, hippocampus, midline, and ventral fit | No change |
| roi4_run4 | 9 → 273 | 9.66 | Excellent; small ventral outliers are not systematic | No change |

## Clean final figure library

The current selected fit for every slice is collected in one clean directory. Historical iteration directories remain audit records and are not final outputs.

![Allen CCF final fit library](../figures/Allen_CCF_alignment_optimized/final/FINAL_FITS_LIBRARY.jpg)

Machine-readable records: `../figures/Allen_CCF_alignment_optimized/final/FINAL_FIGURE_MANIFEST.csv` and `../data/interim/registration/Allen_CCF_regional_tests/all_slices_regional_landmark_final_decisions.csv`.
"""))

## Clean final alignment library

Only the current selected fit is shown for each registered slice. The full-resolution five-figure suite for every slice is in `../figures/Allen_CCF_alignment_optimized/final/`.

![Allen CCF final fit library](../figures/Allen_CCF_alignment_optimized/final/FINAL_FITS_LIBRARY.jpg)

## Selecting finals during the iteration era

> Superseded. This section records how finals were chosen while the sweeps below were the pipeline. `../figures/Allen_CCF_alignment_optimized/final/` now holds the output of the rebuild described at the end of this notebook, and its `README.md` states which run produced it. The historical `iteration_*` directories remain audit records.

Loose legacy PNG/JPG files were removed from the optimized root to prevent stale results from being mistaken for finals. Each slice was rendered with five labeled figures: the `2d reg.png` and `3d reg.png` cell/atlas contour diagnostics, `regional targets.png` for DG, MH, LH, and ependymal populations, `reg_5.jpg` showing fixed, rigid, spline, and Allen-border panels, and `reg_5_cells_all.jpg` showing final Allen borders alone and over all cells.

Under that scheme the accepted regional refinements were `roi1_run2`, `roi2_run2`, and `roi4_run3` from iteration 11, and the other 10 slices kept their retained iteration-03 fit. `FINAL_FIGURE_MANIFEST.csv` records the selection for every file. No canonical figure was modified.

## All-slice regional landmark optimization

The visible red band is `037 DG Glut` (`#CC2400`) and should follow Allen dentate granule-cell layers 632, 758, 790, and 823. The all-slice sweep therefore used explicit DG-blade landmarks, with MH/LH centroids only when supported by both cell count and atlas anatomy. Global AP, rotation, reflection, and scale parameters were retained.

Twenty-seven candidates were evaluated across iterations 11–17: weight 0.03 for the primary pass, 0.08 for initial rescues, 0.01 for over-warped slices, 0.05/0.02/0.04 for under-corrected adaptive tests, and a low-count LH-protection test for `roi4_run1`. Hard gates rejected anchor displacement above 8 × 25 µm voxels, tissue-containment loss, and large losses in protected MH/LH populations.

### Accepted regional refinements

- `roi1_run2`, iteration 11: DG p90 13.93 → 2.00; DG inside 55.6% → 70.0%; MH inside 42.9% → 76.2%; LH inside 10.6% → 92.9%; anchor displacement 5.32.
- `roi2_run2`, iteration 11: DG p90 1.41 → 1.00; DG inside 73.3% → 87.4%; MH inside 59.4% → 65.4%; anchor displacement 6.39.
- `roi4_run3`, iteration 11: DG p90 7.21 → 0.00; DG inside 75.7% → 90.3%; MH inside 46.2% → 66.5%; LH inside 7.3% → 71.4%; anchor displacement 6.06.

### Retained baselines

The other 10 slices retain iteration 03. Their candidates either over-warped the anchored cortex/top surface or failed to improve the intended target safely. In particular, `roi4_run1` iteration 11 was invalidated because LH containment fell from 94.4% to 16.7%; iteration 17 preserved LH but reduced MH from 75.1% to 56.5%, so baseline was retained.

All test parameters, candidate metrics, and final decisions are stored in `../data/interim/registration/Allen_CCF_regional_tests/`. The isolated workbook is `slice_positions_25um_all_slices_regional_landmarks.xlsx`; the final one-row-per-slice table is `all_slices_regional_landmark_final_decisions.csv`. Canonical workbooks, coordinate tables, and figures were not modified.

![All-slice current final fits](../figures/Allen_CCF_alignment_optimized/final/FINAL_FITS_LIBRARY.jpg)

# Reproducible rebuild: `allen_ccf_repro`

The iterative parameter sweeps above were replaced by a single pipeline that decides everything itself and records why. It lives in `python_scripts/allen_ccf_repro/` and runs end to end with one command:

```bash
/resnick/groups/MazmanianLab/jboktor/software/miniforge3/envs/spatialomics/bin/python \
    -m allen_ccf_repro.cli --all --jobs 5 --promote
```

Run from `notebooks/python_scripts/`. Every input, parameter, git commit, and per-slice decision is written to `../data/interim/registration/Allen_CCF_rebuild/run_manifest.json`, and the per-slice numbers to `rebuild_decisions.csv` beside it.

## What the pipeline does for each slice

1. **Region registry.** `configs/region_registry.yaml` freezes the mapping from cell subclass to Allen annotation IDs, and assigns each region a role: `hard` regions drive the fit, `soft` ones inform it, `mask_only` ones are never used as landmarks, and MH/LH are `protected`. CA3 is held out of fitting entirely and used only for validation.
2. **AP level, obliquity, and medial margin search.** One stable affine fit is computed, then candidate atlas planes are scored by re-indexing the annotation through it, so the comparison carries no registration noise. Only the medial margin needs re-registration, because it changes the moving image.
3. **Masks.** Tissue is taken as DAPI signal unioned with cell-supported territory, so blurred-but-present cortex keeps its weight. Bright cell-free blocks become artifacts. Elastix runs unmasked unless a slice has dead zones, in which case the mask is the whole field minus those zones: the background beside intact tissue is what pulls the atlas surface onto the real surface, and removing it measurably degrades the fit.
4. **Landmarks.** DG quantile ladders, CA ribbon medians, MH/LH centroids, and ventricle components, each filtered by cell density so scattered outliers cannot anchor the warp.
5. **Candidate ladder.** A geometry-only fit plus landmark weights 0.03, 0.015, and 0.05. Any fit that folds is retried stiffer.
6. **Gates.** Each candidate is accepted or refused on tissue containment, per-region containment and distance, the mean gain across gated and protected regions, the CA3 holdout, thickness near masks, the Jacobian, and medial clamping. Nothing is accepted on a human eyeballing it.

## Result

Eleven of thirteen slices took a new fit. `roi2_run4` and `roi4_run2` kept their historical ones because no candidate beat them, and their status and figure titles say so. Median containment change across the accepted slices, in percentage points: cortical layer 4/5 +30.6, layer 2/3 +15.9, MH +14.4, VMH +14.1, ventricle walls +11.2, RT +10.1, CA3 +10.0, LH +7.1, choroid plexus +5.9, caudoputamen +4.4, CA1 +4.2, DG +3.9. Tissue containment rose on all thirteen slices, by 1.05 pp at the median and by at least 0.62 pp.

The two slices that previously failed outright are the clearest cases. `roi1_run1` went from CA1 3.2% to 77.8% and caudoputamen 0.1% to 80.0%; `roi2_run1` went from CA1 24.5% to 72.0% and MH 16.9% to 68.0%.

Two ungated populations lose containment at the median, cortical layer 6 by 6.1 pp and deep amygdala by 10.8 pp. Both are boundary flips rather than movement: the median cell-to-region distance stays at zero voxels for each, and the 90th percentile moves only 5.0 to 5.7 voxels for layer 6 and 13.4 to 16.7 for the amygdala. CA2 loses 1.1 pp of containment while its median distance improves from 40.3 to 31.4 voxels, which is what a sliver-sized target does to a binary in-or-out statistic.

## What the rebuild changed about the diagnosis

Several things the earlier iterations treated as defects turned out to be measurement artifacts, and real ones were found in their place.

**The medial pile-up was in the metric, not the fit.** Counting the fraction of habenular cells near the medial crop edge flags any structure that legitimately sits against the midline, which the habenula does. Counting instead the *density* in the edge column relative to its neighbours shows no stacking anywhere: across all thirteen slices that ratio runs 0.27 to 1.17, and 0.00 to 0.67 for the habenular cells specifically, where clamping would give a ratio well above one. The gate refuses anything above 3.0 and never fires. The medial margin is now searched per slice, from 0 to 22 voxels, and the chosen value is recorded. Of the eleven accepted slices only two keep the historical 8-voxel margin; five do better cropped at the midline, one at 4 voxels, and three at 14.

**Regional containment is a brittle statistic on its own.** It is a binary in-or-out test against a structure boundary, so a sub-voxel shift flips a large fraction of the cells packed against that boundary. A containment loss is now only treated as a failure when the median distance also worsens by more than one voxel, which is the resolution of the distance transform, or when the loss is so large the region has clearly slipped off target. The remaining losses on accepted slices are all boundary flips with the median distance unchanged at zero.

**Forgiving each loss separately let a worse fit through.** Because a containment loss with an unchanged median distance is downgraded to a warning, a candidate could collect a dozen such warnings and still be worse overall than the fit it was replacing. `roi4_run2` did exactly that: every one of its losses was individually forgivable, yet it averaged 4.9 pp below its baseline across the gated regions and 9.8 pp below across MH and LH, and it was accepted. The gates now also require the mean gain across gated regions and across protected regions to be non-negative within tolerance, which is what moved `roi4_run2` to a retained baseline.

**Bending-energy regularisation was doing real damage.** It was included to prevent the atlas compressing into missing tissue, but on `roi2_run1` raising the weight from 0 to 0.35 dropped caudoputamen containment from 80% to 4% while the Jacobian stayed positive throughout, so it was buying nothing that the Jacobian gate did not already provide. It is now off by default and applied, together with a coarser control grid, only to a fit that actually folds.

**Per-slice elastix parameters from the sheet matter.** `spline_grid_size`, `iterations`, and `histogram_bins` were tuned per slice for the historical fits. The rebuild now uses them instead of one global setting; combined with dropping the bending penalty this is what recovered `roi2_run1` from 73% to 98% tissue containment.

**Retaining a baseline now really retains it.** A slice whose candidates are all refused writes out its historical coordinates and renders against its historical plane, rather than silently writing the rejected candidate into the output table.

## Missing dorsal tissue

The automatic detector compares the tissue mask against an undistorted (rigid plus uniform scale) atlas placement, since an affine fit absorbs a missing chunk by squeezing and hides it. It found no overhang on any of the thirteen slices, and the layer-labelled cells agree: every section carries a continuous cortical ribbon with layer 2/3 at the surface, rather than deep layers exposed at a cut edge. Localised damage is still visible at the dorsomedial corner of some sections, which no automatic silhouette or cortical-rim test separated from intact sections. Per-slice exclusion polygons can be added to `configs/mask_overrides/<sample>.yaml` and are version-controlled; none are currently supplied, and a slice with a mask is refit at both margins to confirm the decision does not depend on where the mask edge is drawn.